# 🚦 Intelligent Traffic Signal Optimization using TGCN, Transformer and Soft Actor-Critic

### Major Project – I
#### Fallback Implementation

# 1. Import Required Libraries

### Import all Python libraries, deep learning frameworks, graph neural network utilities and reinforcement learning dependencies.

In [81]:
# ==========================================================
# CELL 1 — Mount Google Drive
# ==========================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Configuration and Hyperparameters

### Define project constants, training parameters, device configuration and reproducibility settings.

In [82]:
# ==========================================================
# CELL 2 — Imports
# ==========================================================

%pip install ijson

import os
import json
import math
import random
import ijson

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from collections import Counter

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# 3. Dataset Loading

### Load the traffic dataset and inspect its structure before preprocessing.

In [83]:
# ==========================================================
# CELL 3 — Reproducibility
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


# 4. Data Preprocessing

### Clean, normalize and prepare traffic features for graph-based learning.

In [84]:
# ==========================================================
# CELL 4 — Amazon Dataset Paths
# ==========================================================

BASE_PATH = "/content/drive/MyDrive/amazon_dataset"

EVAL_PATH = os.path.join(
    BASE_PATH,
    "almrrc2021-data-evaluation"
)

APPLY_PATH = os.path.join(
    EVAL_PATH,
    "model_apply_inputs"
)

SCORE_PATH = os.path.join(
    EVAL_PATH,
    "model_score_inputs"
)

ROUTE_FILE = os.path.join(
    APPLY_PATH,
    "eval_route_data.json"
)

PACKAGE_FILE = os.path.join(
    APPLY_PATH,
    "eval_package_data.json"
)

TRAVEL_TIME_FILE = os.path.join(
    APPLY_PATH,
    "eval_travel_times.json"
)

SEQUENCE_FILE = os.path.join(
    SCORE_PATH,
    "eval_actual_sequences.json"
)

INVALID_SEQUENCE_FILE = os.path.join(
    SCORE_PATH,
    "eval_invalid_sequence_scores.json"
)

print("Checking dataset files...")
print("=" * 60)

files = {
    "Route data": ROUTE_FILE,
    "Package data": PACKAGE_FILE,
    "Travel times": TRAVEL_TIME_FILE,
    "Actual sequences": SEQUENCE_FILE,
    "Invalid sequences": INVALID_SEQUENCE_FILE
}

for name, path in files.items():

    exists = os.path.exists(path)

    if exists:
        size = os.path.getsize(path) / (1024 ** 2)
        print(f"✓ {name:<20} {size:,.2f} MB")
    else:
        print(f"✗ {name:<20} NOT FOUND")

Checking dataset files...
✓ Route data           36.03 MB
✓ Package data         158.50 MB
✓ Travel times         804.05 MB
✓ Actual sequences     4.41 MB
✓ Invalid sequences    0.20 MB


# 5. Graph Construction

### Build the road network graph and adjacency matrix representing traffic connectivity.

In [85]:
# ==========================================================
# CELL 5 — Load Amazon Route + Actual Sequence Data
# ==========================================================

print("Loading route data...")

with open(ROUTE_FILE, "r", encoding="utf-8") as f:
    route_data = json.load(f)

print("Route data loaded.")

print("\nLoading actual sequences...")

with open(SEQUENCE_FILE, "r", encoding="utf-8") as f:
    actual_sequences = json.load(f)

print("Actual sequences loaded.")

print("\nDataset summary")
print("=" * 60)

print("Number of routes:", len(route_data))
print("Number of sequences:", len(actual_sequences))

route_ids = set(route_data.keys())
sequence_ids = set(actual_sequences.keys())

print(
    "Matching routes:",
    len(route_ids & sequence_ids)
)

Loading route data...
Route data loaded.

Loading actual sequences...
Actual sequences loaded.

Dataset summary
Number of routes: 3052
Number of sequences: 3052
Matching routes: 3052


# 6. Traffic Environment Initialization

### Create the traffic simulation environment used for reinforcement learning.

In [86]:
# ==========================================================
# CELL 6 — Select Initial Amazon Route
# ==========================================================

ROUTE_ID = (
    "RouteID_00092558-dece-4fb7-8d0d-7d0df3a4864e"
)

assert ROUTE_ID in route_data
assert ROUTE_ID in actual_sequences

route = route_data[ROUTE_ID]
sequence_data = actual_sequences[ROUTE_ID]

print("Selected Route")
print("=" * 60)

print("Route ID:", ROUTE_ID)
print("Station:", route["station_code"])
print("Date:", route["date_YYYY_MM_DD"])
print("Departure:", route["departure_time_utc"])
print("Number of stops:", len(route["stops"]))

Selected Route
Route ID: RouteID_00092558-dece-4fb7-8d0d-7d0df3a4864e
Station: DLA8
Date: 2018-06-17
Departure: 17:23:00
Number of stops: 166


# 7. Dynamic Feature Extraction

### Extract temporal traffic features including vehicle count, queue length and waiting time.

In [87]:
# ==========================================================
# CELL 7 — Build Amazon Actual Route
# ==========================================================

actual_sequence = sequence_data["actual"]

# Convert:
# stop_id -> sequence number
sequence_pairs = list(actual_sequence.items())

sequence_pairs.sort(key=lambda x: x[1])

baseline_rows = []

for sequence_number, (stop_id, amazon_order) in enumerate(sequence_pairs):

    stop_info = route["stops"][stop_id]

    baseline_rows.append({
        "sequence": sequence_number,
        "amazon_order": amazon_order,
        "stop_id": stop_id,
        "type": stop_info["type"],
        "latitude": stop_info["lat"],
        "longitude": stop_info["lng"],
        "zone_id": stop_info.get("zone_id")
    })

baseline_df = pd.DataFrame(baseline_rows)

print("Amazon Baseline Route")
print("=" * 60)

print("Total nodes:", len(baseline_df))
print(
    "Delivery stops:",
    (baseline_df["type"] == "Dropoff").sum()
)

display(baseline_df.head(15))

Amazon Baseline Route
Total nodes: 166
Delivery stops: 165


,sequence,amazon_order,stop_id,type,latitude,longitude,zone_id
0,0,0,UZ,Station,33.918699,-118.324843,NaN
1,1,1,YM,Dropoff,33.892955,-118.346119,K-21.3G
2,2,2,VO,Dropoff,33.891739,-118.346113,K-21.3G
3,3,3,YR,Dropoff,33.891527,-118.347504,K-21.3G
4,4,4,DT,Dropoff,33.891909,-118.347504,K-21.3G
5,5,5,DQ,Dropoff,33.891864,-118.347506,K-21.3G
6,6,6,QT,Dropoff,33.892877,-118.347505,K-21.3G
7,7,7,BO,Dropoff,33.893218,-118.347508,K-21.3G
8,8,8,TM,Dropoff,33.894205,-118.348315,K-21.3G
9,9,9,IV,Dropoff,33.893044,-118.348888,K-21.2G


# 8. Temporal Graph Convolution Network (TGCN)

### Learn spatial relationships between connected intersections using graph convolutions.

In [88]:
# ==========================================================
# CELL 8 — Identify Warehouse / Station
# ==========================================================

station_rows = baseline_df[
    baseline_df["type"].str.lower() == "station"
]

print("Number of stations:", len(station_rows))

if len(station_rows) == 1:

    warehouse_id = station_rows.iloc[0]["stop_id"]

    warehouse_lat = station_rows.iloc[0]["latitude"]
    warehouse_lon = station_rows.iloc[0]["longitude"]

    print("\nWarehouse:")
    print("ID:", warehouse_id)
    print("Latitude:", warehouse_lat)
    print("Longitude:", warehouse_lon)

else:
    print("WARNING: Expected exactly one station.")

Number of stations: 1

Warehouse:
ID: UZ
Latitude: 33.918699
Longitude: -118.324843


# 9. Spatial Feature Generation

### Generate graph-aware node embeddings representing current traffic conditions.

In [89]:
# ==========================================================
# CELL 9 — Extract Travel-Time Matrix for Selected Route
# ==========================================================

print("Searching travel-time file...")
print("Target:", ROUTE_ID)

travel_matrix = None

with open(TRAVEL_TIME_FILE, "rb") as f:

    parser = ijson.kvitems(f, "")

    for route_id, matrix in parser:

        if route_id == ROUTE_ID:
            travel_matrix = matrix
            break

assert travel_matrix is not None

print("\nTravel-time matrix loaded for selected route.")

print("Number of origins:", len(travel_matrix))

first_origin = next(iter(travel_matrix))

print("First origin:", first_origin)

print(
    "Destinations from first origin:",
    len(travel_matrix[first_origin])
)

Searching travel-time file...
Target: RouteID_00092558-dece-4fb7-8d0d-7d0df3a4864e

Travel-time matrix loaded for selected route.
Number of origins: 166
First origin: AH
Destinations from first origin: 166


# 10. Baseline Temporal Modeling (LSTM)

### Learn temporal traffic patterns from TGCN embeddings.

In [90]:
# ==========================================================
# CELL 10 — Build Real Amazon Graph
# ==========================================================

# Get all nodes from the selected route
node_ids = list(route["stops"].keys())

print("Number of Amazon nodes:", len(node_ids))

# Create node -> integer index mapping
node_to_idx = {
    node_id: idx
    for idx, node_id in enumerate(node_ids)
}

idx_to_node = {
    idx: node_id
    for node_id, idx in node_to_idx.items()
}

# Verify that travel-time matrix contains the same nodes
travel_nodes = set(travel_matrix.keys())
route_nodes = set(node_ids)

print("Nodes in route data:", len(route_nodes))
print("Nodes in travel-time matrix:", len(travel_nodes))

print(
    "Common nodes:",
    len(route_nodes.intersection(travel_nodes))
)

print(
    "Route nodes missing from travel matrix:",
    len(route_nodes - travel_nodes)
)

print(
    "Travel nodes missing from route:",
    len(travel_nodes - route_nodes)
)

# ----------------------------------------------------------
# Create travel-time adjacency matrix
# ----------------------------------------------------------

N = len(node_ids)

travel_time_matrix = np.zeros((N, N), dtype=np.float32)

for origin in node_ids:

    origin_idx = node_to_idx[origin]

    for destination, travel_time in travel_matrix[origin].items():

        if destination in node_to_idx:

            destination_idx = node_to_idx[destination]

            travel_time_matrix[
                origin_idx,
                destination_idx
            ] = float(travel_time)

print("\nTravel-time matrix shape:")
print(travel_time_matrix.shape)

print("\nExample:")
example_origin = node_ids[0]
example_destination = node_ids[1]

print(
    f"{example_origin} -> {example_destination}: "
    f"{travel_time_matrix[0, 1]:.2f} seconds"
)

Number of Amazon nodes: 166
Nodes in route data: 166
Nodes in travel-time matrix: 166
Common nodes: 166
Route nodes missing from travel matrix: 0
Travel nodes missing from route: 0

Travel-time matrix shape:
(166, 166)

Example:
AH -> AJ: 484.50 seconds


# 11. PPO Agent

### Train the Proximal Policy Optimization agent to optimize traffic signal timing.

In [91]:
# ==========================================================
# CELL 11 — Construct GCN Adjacency from Real Travel Times
# ==========================================================

# Copy travel-time matrix
T = travel_time_matrix.copy()

# ----------------------------------------------------------
# Remove self-travel times
# ----------------------------------------------------------

non_zero_times = T[T > 0]

print("Non-zero OD pairs:", len(non_zero_times))

# ----------------------------------------------------------
# Calculate scale parameter
# ----------------------------------------------------------

tau = float(np.median(non_zero_times))

print("Median travel time (tau):")
print(f"{tau:.2f} seconds")

# ----------------------------------------------------------
# Convert travel time -> graph connection strength
#
# Short travel time  -> strong connection
# Long travel time   -> weak connection
# ----------------------------------------------------------

A = np.zeros_like(T, dtype=np.float32)

mask = T > 0

A[mask] = np.exp(-T[mask] / tau)

# ----------------------------------------------------------
# Add self-loops
# ----------------------------------------------------------

np.fill_diagonal(A, 1.0)

# ----------------------------------------------------------
# Inspect adjacency
# ----------------------------------------------------------

print("\nAdjacency matrix shape:")
print(A.shape)

print("\nAdjacency statistics")
print("=" * 60)

print("Minimum:", A.min())
print("Maximum:", A.max())
print("Mean:", A.mean())

print("\nExample connections:")

for j in range(1, 6):

    print(
        f"{idx_to_node[0]} -> {idx_to_node[j]} | "
        f"Travel time: {T[0,j]:.2f}s | "
        f"Graph weight: {A[0,j]:.4f}"
    )

Non-zero OD pairs: 27384
Median travel time (tau):
246.90 seconds

Adjacency matrix shape:
(166, 166)

Adjacency statistics
Minimum: 0.0
Maximum: 1.0
Mean: 0.40152818

Example connections:
AH -> AJ | Travel time: 484.50s | Graph weight: 0.1405
AH -> AL | Travel time: 399.30s | Graph weight: 0.1984
AH -> AN | Travel time: 216.70s | Graph weight: 0.4157
AH -> AP | Travel time: 94.10s | Graph weight: 0.6831
AH -> AS | Travel time: 132.90s | Graph weight: 0.5838


# 12. Model Training

### Train the complete TGCN–LSTM–PPO pipeline.

In [92]:
# ==========================================================
# CELL 12 — Normalize Amazon Graph for GCN
# ==========================================================

# Convert adjacency to float32
A_tensor = torch.tensor(
    A,
    dtype=torch.float32
)

# ----------------------------------------------------------
# Degree matrix
# ----------------------------------------------------------

degree = A_tensor.sum(dim=1)

print("Degree statistics")
print("=" * 60)

print("Minimum degree:", degree.min().item())
print("Maximum degree:", degree.max().item())
print("Mean degree:", degree.mean().item())

# ----------------------------------------------------------
# D^(-1/2)
# ----------------------------------------------------------

degree_inv_sqrt = torch.pow(
    degree,
    -0.5
)

# Numerical safety
degree_inv_sqrt[
    torch.isinf(degree_inv_sqrt)
] = 0.0

D_inv_sqrt = torch.diag(degree_inv_sqrt)

# ----------------------------------------------------------
# Symmetric normalization
# ----------------------------------------------------------

A_normalized = (
    D_inv_sqrt
    @ A_tensor
    @ D_inv_sqrt
)

print("\nNormalized adjacency shape:")
print(A_normalized.shape)

print("\nNormalized adjacency statistics")
print("=" * 60)

print("Minimum:", A_normalized.min().item())
print("Maximum:", A_normalized.max().item())
print("Mean:", A_normalized.mean().item())

# Verify numerical validity
print("\nContains NaN:",
      torch.isnan(A_normalized).any().item())

print("Contains Inf:",
      torch.isinf(A_normalized).any().item())

Degree statistics
Minimum degree: 9.762850761413574
Maximum degree: 85.98299407958984
Mean degree: 66.6536865234375

Normalized adjacency shape:
torch.Size([166, 166])

Normalized adjacency statistics
Minimum: 0.0
Maximum: 0.1024291068315506
Mean: 0.005984483286738396

Contains NaN: False
Contains Inf: False


# 13. Model Evaluation

### Evaluate traffic optimization performance using multiple traffic metrics.

In [93]:
# ==========================================================
# CELL 13 — Build Real Amazon Node Features
# ==========================================================

# ----------------------------------------------------------
# Extract coordinates
# ----------------------------------------------------------

coordinates = []

for node_id in node_ids:

    stop_info = route["stops"][node_id]

    coordinates.append([
        float(stop_info["lat"]),
        float(stop_info["lng"])
    ])

coordinates = np.array(
    coordinates,
    dtype=np.float32
)

# ----------------------------------------------------------
# Normalize coordinates
# ----------------------------------------------------------

lat = coordinates[:, 0]
lon = coordinates[:, 1]

lat_min, lat_max = lat.min(), lat.max()
lon_min, lon_max = lon.min(), lon.max()

lat_norm = (
    (lat - lat_min) /
    (lat_max - lat_min + 1e-8)
)

lon_norm = (
    (lon - lon_min) /
    (lon_max - lon_min + 1e-8)
)

# ----------------------------------------------------------
# Node type features
# ----------------------------------------------------------

warehouse_feature = np.array([
    1.0 if node_id == warehouse_id else 0.0
    for node_id in node_ids
], dtype=np.float32)

delivery_feature = np.array([
    1.0 if route["stops"][node_id]["type"].lower() == "dropoff"
    else 0.0
    for node_id in node_ids
], dtype=np.float32)

# ----------------------------------------------------------
# Combine features
# ----------------------------------------------------------

node_features = np.column_stack([
    lat_norm,
    lon_norm,
    warehouse_feature,
    delivery_feature
]).astype(np.float32)

# ----------------------------------------------------------
# Convert to PyTorch tensor
# ----------------------------------------------------------

X = torch.tensor(
    node_features,
    dtype=torch.float32
)

print("Node feature matrix shape:")
print(X.shape)

print("\nFeature definitions:")
print("0 = normalized latitude")
print("1 = normalized longitude")
print("2 = warehouse indicator")
print("3 = delivery indicator")

print("\nFirst 10 nodes:")
print(
    pd.DataFrame(
        X.numpy(),
        columns=[
            "lat_norm",
            "lon_norm",
            "warehouse",
            "delivery"
        ]
    ).head(10)
)

Node feature matrix shape:
torch.Size([166, 4])

Feature definitions:
0 = normalized latitude
1 = normalized longitude
2 = warehouse indicator
3 = delivery indicator

First 10 nodes:
   lat_norm  lon_norm  warehouse  delivery
0  0.173814  0.459510        0.0       1.0
1  0.604839  0.166771        0.0       1.0
2  0.376186  0.087675        0.0       1.0
3  0.320493  0.303201        0.0       1.0
4  0.040417  0.460556        0.0       1.0
5  0.197628  0.312199        0.0       1.0
6  0.173909  0.392760        0.0       1.0
7  0.040512  0.433145        0.0       1.0
8  0.503985  0.302364        0.0       1.0
9  0.366319  0.378322        0.0       1.0


# 14. Visualization

### Plot training rewards, traffic statistics and model performance.

In [94]:
# ==========================================================
# CELL 14 — Real Amazon GCN
# ==========================================================

class GCNLayer(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(
            in_features,
            out_features
        )

    def forward(self, X, A):
        """
        X: Node features
           [num_nodes, in_features]

        A: Normalized adjacency
           [num_nodes, num_nodes]
        """

        # Graph message passing
        H = torch.matmul(A, X)

        # Learnable transformation
        H = self.linear(H)

        # Non-linearity
        H = F.relu(H)

        return H


class AmazonGCN(nn.Module):

    def __init__(
        self,
        input_features=4,
        hidden_features=16,
        output_features=16
    ):
        super().__init__()

        self.gcn1 = GCNLayer(
            input_features,
            hidden_features
        )

        self.gcn2 = GCNLayer(
            hidden_features,
            output_features
        )

    def forward(self, X, A):

        H = self.gcn1(X, A)

        H = self.gcn2(H, A)

        return H


# ----------------------------------------------------------
# Create model
# ----------------------------------------------------------

gcn_model = AmazonGCN(
    input_features=4,
    hidden_features=16,
    output_features=16
)

print(gcn_model)

AmazonGCN(
  (gcn1): GCNLayer(
    (linear): Linear(in_features=4, out_features=16, bias=True)
  )
  (gcn2): GCNLayer(
    (linear): Linear(in_features=16, out_features=16, bias=True)
  )
)


---
# 🔄 Fallback Architecture

### Alternative implementation replacing the LSTM + PPO pipeline with Transformer + Soft Actor-Critic while preserving the remaining system.
---

# 15. Transformer Encoder

### Replace the LSTM with a Transformer Encoder to capture long-range temporal dependencies using self-attention.

In [95]:
# ==========================================================
# CELL 15 — Test GCN on Real Amazon Route
# ==========================================================

gcn_model.eval()

with torch.no_grad():

    node_embeddings = gcn_model(
        X,
        A_normalized
    )

print("Input feature shape:")
print(X.shape)

print("\nGCN embedding shape:")
print(node_embeddings.shape)

print("\nFirst node embedding:")
print(node_embeddings[0])

print("\nEmbedding statistics")
print("=" * 60)

print("Mean:", node_embeddings.mean().item())
print("Std:", node_embeddings.std().item())
print("Minimum:", node_embeddings.min().item())
print("Maximum:", node_embeddings.max().item())

Input feature shape:
torch.Size([166, 4])

GCN embedding shape:
torch.Size([166, 16])

First node embedding:
tensor([0.0000, 0.2005, 0.0000, 0.0000, 0.0107, 0.1769, 0.0000, 0.0000, 0.0000,
        0.0000, 0.1244, 0.0000, 0.0715, 0.0279, 0.0000, 0.0000])

Embedding statistics
Mean: 0.0388081856071949
Std: 0.06650104373693466
Minimum: 0.0
Maximum: 0.2136755734682083


# 16. Positional Encoding

### Encode temporal order information required by the Transformer architecture.

In [96]:
# ==========================================================
# CELL 16 — Real Amazon Travel-Time Node State
# ==========================================================

# ----------------------------------------------------------
# Remove self-travel times
# ----------------------------------------------------------

T_state = travel_time_matrix.copy()

# Self travel times should not influence statistics
np.fill_diagonal(T_state, np.nan)

# ----------------------------------------------------------
# Calculate node-level travel-time statistics
# ----------------------------------------------------------

node_mean_time = np.nanmean(
    np.where(T_state > 0, T_state, np.nan),
    axis=1
)

node_median_time = np.nanmedian(
    np.where(T_state > 0, T_state, np.nan),
    axis=1
)

node_min_time = np.nanmin(
    np.where(T_state > 0, T_state, np.nan),
    axis=1
)

node_max_time = np.nanmax(
    np.where(T_state > 0, T_state, np.nan),
    axis=1
)

# ----------------------------------------------------------
# Normalize each statistic
# ----------------------------------------------------------

def min_max_normalize(values):

    v_min = np.nanmin(values)
    v_max = np.nanmax(values)

    return (
        (values - v_min) /
        (v_max - v_min + 1e-8)
    )


mean_norm = min_max_normalize(node_mean_time)
median_norm = min_max_normalize(node_median_time)
min_norm = min_max_normalize(node_min_time)
max_norm = min_max_normalize(node_max_time)

# ----------------------------------------------------------
# Build travel-time state
# ----------------------------------------------------------

travel_time_features = np.column_stack([
    mean_norm,
    median_norm,
    min_norm,
    max_norm
]).astype(np.float32)

travel_time_state = torch.tensor(
    travel_time_features,
    dtype=torch.float32
)

print("Travel-time state shape:")
print(travel_time_state.shape)

print("\nFeature definitions:")
print("0 = normalized mean outgoing travel time")
print("1 = normalized median outgoing travel time")
print("2 = normalized minimum outgoing travel time")
print("3 = normalized maximum outgoing travel time")

print("\nFirst 10 nodes:")
print(
    pd.DataFrame(
        travel_time_state.numpy(),
        columns=[
            "mean_time",
            "median_time",
            "min_time",
            "max_time"
        ]
    ).head(10)
)

Travel-time state shape:
torch.Size([166, 4])

Feature definitions:
0 = normalized mean outgoing travel time
1 = normalized median outgoing travel time
2 = normalized minimum outgoing travel time
3 = normalized maximum outgoing travel time

First 10 nodes:
   mean_time  median_time  min_time  max_time
0   0.149915     0.115308  0.029119  0.456664
1   0.249864     0.220084  0.022727  0.660764
2   0.142021     0.111494  0.018111  0.717925
3   0.048158     0.044670  0.024858  0.507922
4   0.232471     0.241874  0.018999  0.637776
5   0.102719     0.077719  0.043857  0.624417
6   0.147815     0.136917  0.012074  0.611370
7   0.242077     0.247503  0.017578  0.671326
8   0.113091     0.092065  0.084517  0.329916
9   0.068770     0.084075  0.012429  0.292948


# 17. Soft Actor-Critic (SAC)

### Implement the Soft Actor-Critic agent for continuous traffic signal control.

In [97]:
# ==========================================================
# CELL 17 — Combine GCN + Travel-Time Features
# ==========================================================

# GCN embeddings from Cell 15
spatial_features = node_embeddings

# Travel-time state from Cell 16
traffic_features = travel_time_state

print("Spatial feature shape:")
print(spatial_features.shape)

print("\nTravel-time feature shape:")
print(traffic_features.shape)

# ----------------------------------------------------------
# Combine features
# ----------------------------------------------------------

combined_node_features = torch.cat(
    [
        spatial_features,
        traffic_features
    ],
    dim=1
)

print("\nCombined node feature shape:")
print(combined_node_features.shape)

print("\nExpected:")
print("166 nodes × 20 features")

Spatial feature shape:
torch.Size([166, 16])

Travel-time feature shape:
torch.Size([166, 4])

Combined node feature shape:
torch.Size([166, 20])

Expected:
166 nodes × 20 features


# 18. Replay Buffer

### Store and reuse experience tuples for off-policy reinforcement learning.

In [98]:
# ==========================================================
# CELL 18 — TGCN + LSTM Temporal Encoder
# ==========================================================

class TemporalEncoder(nn.Module):

    def __init__(
        self,
        input_size=20,
        hidden_size=32,
        num_layers=1
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, x):

        # x:
        # [batch, sequence_length, input_size]

        output, (hidden, cell) = self.lstm(x)

        # Return final hidden state
        return hidden[-1]


# ----------------------------------------------------------
# Create temporal encoder
# ----------------------------------------------------------

temporal_encoder = TemporalEncoder(
    input_size=20,
    hidden_size=32
)

print(temporal_encoder)


TemporalEncoder(
  (lstm): LSTM(20, 32, batch_first=True)
)


# 19. Transformer + SAC Integration

### Connect TGCN spatial embeddings with the Transformer and SAC policy network.

In [99]:
# ==========================================================
# CELL 19 — Create Initial Route-Level State
# ==========================================================

# Aggregate node features across the route
route_state = combined_node_features.mean(
    dim=0,
    keepdim=True
)

print("Route state before sequence dimension:")
print(route_state.shape)

# ----------------------------------------------------------
# Add sequence dimension
# ----------------------------------------------------------

# [1, 20] -> [1, 1, 20]

lstm_input = route_state.unsqueeze(1)

print("\nLSTM input shape:")
print(lstm_input.shape)

# ----------------------------------------------------------
# Pass through LSTM
# ----------------------------------------------------------

temporal_encoder.eval()

with torch.no_grad():

    temporal_embedding = temporal_encoder(
        lstm_input
    )

print("\nTemporal embedding shape:")
print(temporal_embedding.shape)

print("\nTemporal embedding:")
print(temporal_embedding)

Route state before sequence dimension:
torch.Size([1, 20])

LSTM input shape:
torch.Size([1, 1, 20])

Temporal embedding shape:
torch.Size([1, 32])

Temporal embedding:
tensor([[-0.0401,  0.0252, -0.0016,  0.0745,  0.0120,  0.0015, -0.0725, -0.0083,
          0.0562,  0.0215, -0.0280,  0.0384,  0.0301, -0.0052, -0.0183,  0.0165,
         -0.0387,  0.0268, -0.0078, -0.0248, -0.0493, -0.0478, -0.0214, -0.0424,
          0.0525,  0.0488, -0.0407, -0.0757, -0.0124,  0.0510, -0.0735,  0.0128]])


# 20. Fallback Model Execution

### Execute the complete TGCN → Transformer → SAC pipeline.

In [100]:
# ==========================================================
# CELL 20 — Dynamic Amazon Routing Environment
# ==========================================================

class AmazonRoutingEnvironment:

    def __init__(
        self,
        node_ids,
        travel_time_matrix,
        warehouse_id,
        actual_sequence
    ):

        self.node_ids = node_ids
        self.node_to_idx = {
            node_id: i
            for i, node_id in enumerate(node_ids)
        }

        self.idx_to_node = {
            i: node_id
            for node_id, i in self.node_to_idx.items()
        }

        self.travel_time_matrix = travel_time_matrix

        self.warehouse_id = warehouse_id
        self.warehouse_idx = self.node_to_idx[warehouse_id]

        # Amazon actual sequence
        self.actual_sequence = actual_sequence

        # Number of nodes
        self.num_nodes = len(node_ids)

        self.reset()

    def reset(self):

        # Current location starts at warehouse
        self.current_node = self.warehouse_idx

        # Visited nodes
        self.visited = np.zeros(
            self.num_nodes,
            dtype=np.float32
        )

        # Warehouse is considered visited
        self.visited[self.warehouse_idx] = 1.0

        # Accumulated route time
        self.total_time = 0.0

        # Number of deliveries completed
        self.deliveries_completed = 0

        # History of routing decisions
        self.history = []

        return self.get_state()

    def get_available_actions(self):

        # All unvisited delivery nodes
        available = np.where(
            self.visited == 0
        )[0]

        return available

    def get_state(self):

        return {
            "current_node": self.current_node,
            "current_stop": self.idx_to_node[
                self.current_node
            ],
            "visited": self.visited.copy(),
            "available_actions": self.get_available_actions(),
            "total_time": self.total_time,
            "deliveries_completed": self.deliveries_completed
        }


# ----------------------------------------------------------
# Create environment
# ----------------------------------------------------------

env = AmazonRoutingEnvironment(
    node_ids=node_ids,
    travel_time_matrix=travel_time_matrix,
    warehouse_id=warehouse_id,
    actual_sequence=actual_sequence
)

state = env.reset()

print("Environment created successfully")
print("=" * 60)

print("Number of nodes:", env.num_nodes)

print(
    "Warehouse:",
    env.warehouse_id
)

print(
    "Starting node:",
    state["current_stop"]
)

print(
    "Available actions:",
    len(state["available_actions"])
)

print(
    "Visited nodes:",
    int(state["visited"].sum())
)

print(
    "Deliveries completed:",
    state["deliveries_completed"]
)

print(
    "Total route time:",
    state["total_time"]
)

Environment created successfully
Number of nodes: 166
Warehouse: UZ
Starting node: UZ
Available actions: 165
Visited nodes: 1
Deliveries completed: 0
Total route time: 0.0


# 21. Fallback Evaluation

### Evaluate the alternative architecture using the same traffic performance metrics.

In [101]:
# ==========================================================
# CELL 21 — Add Step Function to Routing Environment
# ==========================================================

class AmazonRoutingEnvironment:

    def __init__(
        self,
        node_ids,
        travel_time_matrix,
        warehouse_id,
        actual_sequence
    ):

        self.node_ids = node_ids

        self.node_to_idx = {
            node_id: i
            for i, node_id in enumerate(node_ids)
        }

        self.idx_to_node = {
            i: node_id
            for node_id, i in self.node_to_idx.items()
        }

        self.travel_time_matrix = travel_time_matrix

        self.warehouse_id = warehouse_id

        self.warehouse_idx = self.node_to_idx[
            warehouse_id
        ]

        self.actual_sequence = actual_sequence

        self.num_nodes = len(node_ids)

        self.reset()

    # ------------------------------------------------------
    # RESET
    # ------------------------------------------------------

    def reset(self):

        self.current_node = self.warehouse_idx

        self.visited = np.zeros(
            self.num_nodes,
            dtype=np.float32
        )

        # Warehouse is already visited
        self.visited[self.warehouse_idx] = 1.0

        self.total_time = 0.0

        self.deliveries_completed = 0

        self.history = [
            self.current_node
        ]

        return self.get_state()

    # ------------------------------------------------------
    # AVAILABLE ACTIONS
    # ------------------------------------------------------

    def get_available_actions(self):

        available = np.where(
            self.visited == 0
        )[0]

        return available

    # ------------------------------------------------------
    # STATE
    # ------------------------------------------------------

    def get_state(self):

        return {
            "current_node": self.current_node,

            "current_stop":
                self.idx_to_node[
                    self.current_node
                ],

            "visited":
                self.visited.copy(),

            "available_actions":
                self.get_available_actions(),

            "total_time":
                self.total_time,

            "deliveries_completed":
                self.deliveries_completed
        }

    # ------------------------------------------------------
    # STEP
    # ------------------------------------------------------

    def step(self, action):

        # ----------------------------------------------
        # Validate action
        # ----------------------------------------------

        action = int(action)

        if action < 0 or action >= self.num_nodes:
            raise ValueError(
                f"Invalid action index: {action}"
            )

        if self.visited[action] == 1:
            raise ValueError(
                f"Node {self.idx_to_node[action]} "
                f"has already been visited."
            )

        # ----------------------------------------------
        # Current -> selected destination
        # ----------------------------------------------

        current = self.current_node

        travel_time = float(
            self.travel_time_matrix[
                current,
                action
            ]
        )

        # ----------------------------------------------
        # Update environment
        # ----------------------------------------------

        self.total_time += travel_time

        self.current_node = action

        self.visited[action] = 1.0

        self.deliveries_completed += 1

        self.history.append(action)

        # ----------------------------------------------
        # Check termination
        # ----------------------------------------------

        done = (
            self.deliveries_completed
            ==
            self.num_nodes - 1
        )

        # ----------------------------------------------
        # Basic reward
        #
        # Lower travel time = higher reward
        # ----------------------------------------------

        reward = -travel_time

        # ----------------------------------------------
        # New state
        # ----------------------------------------------

        next_state = self.get_state()

        return (
            next_state,
            reward,
            done
        )


# ==========================================================
# CREATE ENVIRONMENT
# ==========================================================

env = AmazonRoutingEnvironment(
    node_ids=node_ids,
    travel_time_matrix=travel_time_matrix,
    warehouse_id=warehouse_id,
    actual_sequence=actual_sequence
)

state = env.reset()

print("Environment initialized")
print("=" * 60)

print("Starting node:",
      state["current_stop"])

print("Available actions:",
      len(state["available_actions"]))

print("Total time:",
      state["total_time"])

Environment initialized
Starting node: UZ
Available actions: 165
Total time: 0.0


In [102]:
# ==========================================================
# TEST ONE ROUTING ACTION
# ==========================================================

first_action = state["available_actions"][0]

destination = env.idx_to_node[first_action]

print("Current node:")
print(env.idx_to_node[env.current_node])

print("\nSelected destination:")
print(destination)

next_state, reward, done = env.step(
    first_action
)

print("\nAfter step")
print("=" * 60)

print("New current node:",
      next_state["current_stop"])

print("Travel time added:",
      -reward,
      "seconds")

print("Total route time:",
      next_state["total_time"])

print("Deliveries completed:",
      next_state["deliveries_completed"])

print("Remaining actions:",
      len(next_state["available_actions"]))

print("Done:",
      done)

Current node:
UZ

Selected destination:
AH

After step
New current node: AH
Travel time added: 703.9000244140625 seconds
Total route time: 703.9000244140625
Deliveries completed: 1
Remaining actions: 164
Done: False


# 22. Comparative Analysis

### Compare the original TGCN–LSTM–PPO pipeline with the proposed TGCN–Transformer–SAC fallback architecture.

In [103]:
# ==========================================================
# CELL 22 — Dynamic Routing State Encoder
# ==========================================================

class DynamicStateEncoder(nn.Module):

    def __init__(
        self,
        gcn_model,
        temporal_encoder
    ):
        super().__init__()

        self.gcn = gcn_model
        self.temporal_encoder = temporal_encoder

    def forward(
        self,
        node_features,
        travel_features,
        visited,
        current_node,
        adjacency
    ):

        # --------------------------------------------------
        # GCN spatial representation
        # --------------------------------------------------

        spatial = self.gcn(
            node_features,
            adjacency
        )

        # --------------------------------------------------
        # Convert routing state to tensors
        # --------------------------------------------------

        visited_tensor = torch.tensor(
            visited,
            dtype=torch.float32,
            device=spatial.device
        ).unsqueeze(1)

        current_tensor = torch.zeros(
            spatial.shape[0],
            1,
            dtype=torch.float32,
            device=spatial.device
        )

        current_tensor[current_node] = 1.0

        # --------------------------------------------------
        # Combine node-level features
        # --------------------------------------------------

        dynamic_features = torch.cat(
            [
                spatial,
                travel_features,
                visited_tensor,
                current_tensor
            ],
            dim=1
        )

        return dynamic_features


# ==========================================================
# CREATE ENCODER
# ==========================================================

dynamic_encoder = DynamicStateEncoder(
    gcn_model=gcn_model,
    temporal_encoder=temporal_encoder
)

print(dynamic_encoder)

DynamicStateEncoder(
  (gcn): AmazonGCN(
    (gcn1): GCNLayer(
      (linear): Linear(in_features=4, out_features=16, bias=True)
    )
    (gcn2): GCNLayer(
      (linear): Linear(in_features=16, out_features=16, bias=True)
    )
  )
  (temporal_encoder): TemporalEncoder(
    (lstm): LSTM(20, 32, batch_first=True)
  )
)


In [104]:
# ==========================================================
# CELL 23 — Update LSTM for Dynamic State
# ==========================================================

temporal_encoder = TemporalEncoder(
    input_size=22,
    hidden_size=32
)

print("Updated Temporal Encoder:")
print(temporal_encoder)

Updated Temporal Encoder:
TemporalEncoder(
  (lstm): LSTM(22, 32, batch_first=True)
)


In [105]:
# ==========================================================
# CELL 24 — Generate Dynamic State
# ==========================================================

# Reset environment
state = env.reset()

# ----------------------------------------------------------
# Generate dynamic node representation
# ----------------------------------------------------------

dynamic_node_features = dynamic_encoder(
    node_features=X,
    travel_features=travel_time_state,
    visited=state["visited"],
    current_node=state["current_node"],
    adjacency=A_normalized
)

print("Dynamic node feature shape:")
print(dynamic_node_features.shape)

print("\nExpected:")
print("(166, 22)")

print("\nContains NaN:",
      torch.isnan(dynamic_node_features).any().item())

print("Contains Inf:",
      torch.isinf(dynamic_node_features).any().item())

Dynamic node feature shape:
torch.Size([166, 22])

Expected:
(166, 22)

Contains NaN: False
Contains Inf: False


In [106]:
# ==========================================================
# CELL 25 — Create Initial LSTM State
# ==========================================================

# ----------------------------------------------------------
# Aggregate node states into one route-level state
# ----------------------------------------------------------

route_dynamic_state = dynamic_node_features.mean(
    dim=0,
    keepdim=True
)

print("Route dynamic state:")
print(route_dynamic_state.shape)

# ----------------------------------------------------------
# Add sequence dimension
# ----------------------------------------------------------

lstm_input = route_dynamic_state.unsqueeze(1)

print("\nLSTM input:")
print(lstm_input.shape)

# ----------------------------------------------------------
# Run through LSTM
# ----------------------------------------------------------

temporal_encoder.eval()

with torch.no_grad():

    temporal_state = temporal_encoder(
        lstm_input
    )

print("\nTemporal state:")
print(temporal_state.shape)

print("\nExpected:")
print("torch.Size([1, 32])")

print("\nContains NaN:",
      torch.isnan(temporal_state).any().item())

print("Contains Inf:",
      torch.isinf(temporal_state).any().item())

Route dynamic state:
torch.Size([1, 22])

LSTM input:
torch.Size([1, 1, 22])

Temporal state:
torch.Size([1, 32])

Expected:
torch.Size([1, 32])

Contains NaN: False
Contains Inf: False


In [107]:
# ==========================================================
# CELL 26 — PPO Actor-Critic Network
# ==========================================================

class PPOActorCritic(nn.Module):

    def __init__(
        self,
        state_dim=32,
        num_actions=166,
        hidden_dim=128
    ):
        super().__init__()

        # --------------------------------------------------
        # Shared feature representation
        # --------------------------------------------------

        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # --------------------------------------------------
        # Actor
        # --------------------------------------------------

        self.actor = nn.Linear(
            hidden_dim,
            num_actions
        )

        # --------------------------------------------------
        # Critic
        # --------------------------------------------------

        self.critic = nn.Linear(
            hidden_dim,
            1
        )

    def forward(
        self,
        state,
        action_mask=None
    ):

        # Shared representation
        features = self.shared(state)

        # Actor logits
        logits = self.actor(features)

        # --------------------------------------------------
        # Mask unavailable actions
        # --------------------------------------------------

        if action_mask is not None:

            logits = logits.masked_fill(
                action_mask == 0,
                -1e9
            )

        # Critic value
        value = self.critic(features)

        return logits, value


# ==========================================================
# CREATE PPO MODEL
# ==========================================================

ppo_model = PPOActorCritic(
    state_dim=32,
    num_actions=166,
    hidden_dim=128
)

print(ppo_model)

PPOActorCritic(
  (shared): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
  )
  (actor): Linear(in_features=128, out_features=166, bias=True)
  (critic): Linear(in_features=128, out_features=1, bias=True)
)


In [108]:
# ==========================================================
# CELL 27 — PPO Action Selection with Action Masking
# ==========================================================

# ----------------------------------------------------------
# Prepare current LSTM state
# ----------------------------------------------------------

state_vector = temporal_state

print("Temporal state shape:")
print(state_vector.shape)

# ----------------------------------------------------------
# Create action mask
# 1 = available
# 0 = unavailable
# ----------------------------------------------------------

action_mask = torch.tensor(
    1.0 - state["visited"],
    dtype=torch.float32
).unsqueeze(0)

print("\nAction mask shape:")
print(action_mask.shape)

print(
    "Available actions:",
    int(action_mask.sum().item())
)

# ----------------------------------------------------------
# Actor-Critic forward pass
# ----------------------------------------------------------

ppo_model.eval()

with torch.no_grad():

    logits, value = ppo_model(
        state_vector,
        action_mask
    )

print("\nActor logits shape:")
print(logits.shape)

print("Critic value shape:")
print(value.shape)

# ----------------------------------------------------------
# Convert logits to probability distribution
# ----------------------------------------------------------

dist = torch.distributions.Categorical(
    logits=logits
)

# Sample one action
action = dist.sample()

# Log probability
log_probability = dist.log_prob(action)

# ----------------------------------------------------------
# Results
# ----------------------------------------------------------

action_idx = action.item()

print("\nPPO Decision")
print("=" * 60)

print(
    "Selected action index:",
    action_idx
)

print(
    "Selected stop:",
    env.idx_to_node[action_idx]
)

print(
    "Probability:",
    dist.probs[0, action_idx].item()
)

print(
    "Log probability:",
    log_probability.item()
)

print(
    "Critic value:",
    value.item()
)

# ----------------------------------------------------------
# Verify masking
# ----------------------------------------------------------

visited_indices = np.where(
    state["visited"] == 1
)[0]

masked_probabilities = dist.probs[
    0,
    visited_indices
]

print("\nVisited-node probabilities:")
print(masked_probabilities)

print(
    "\nMaximum probability assigned "
    "to visited nodes:",
    masked_probabilities.max().item()
)

Temporal state shape:
torch.Size([1, 32])

Action mask shape:
torch.Size([1, 166])
Available actions: 165

Actor logits shape:
torch.Size([1, 166])
Critic value shape:
torch.Size([1, 1])

PPO Decision
Selected action index: 138
Selected stop: WQ
Probability: 0.005876489449292421
Log probability: -5.136795997619629
Critic value: -0.04983910173177719

Visited-node probabilities:
tensor([0.])

Maximum probability assigned to visited nodes: 0.0


In [109]:
# ==========================================================
# CELL 28 — Execute PPO Action in Amazon Environment
# ==========================================================

# ----------------------------------------------------------
# Execute selected action
# ----------------------------------------------------------

next_state, reward, done = env.step(
    action_idx
)

# ----------------------------------------------------------
# Display transition
# ----------------------------------------------------------

print("PPO ROUTING TRANSITION")
print("=" * 60)

print(
    "Previous node:",
    state["current_stop"]
)

print(
    "Selected node:",
    env.idx_to_node[action_idx]
)

print(
    "Travel time:",
    -reward,
    "seconds"
)

print(
    "Reward:",
    reward
)

print(
    "New current node:",
    next_state["current_stop"]
)

print(
    "Total route time:",
    next_state["total_time"],
    "seconds"
)

print(
    "Deliveries completed:",
    next_state["deliveries_completed"]
)

print(
    "Remaining actions:",
    len(next_state["available_actions"])
)

print(
    "Episode finished:",
    done
)

PPO ROUTING TRANSITION
Previous node: UZ
Selected node: WQ
Travel time: 840.2000122070312 seconds
Reward: -840.2000122070312
New current node: WQ
Total route time: 840.2000122070312 seconds
Deliveries completed: 1
Remaining actions: 164
Episode finished: False


In [110]:
# ==========================================================
# CELL 29 — PPO Trajectory Collector
# ==========================================================

def collect_trajectory(
    env,
    ppo_model,
    dynamic_encoder,
    temporal_encoder,
    node_features,
    travel_features,
    adjacency,
    max_steps=None
):

    # ------------------------------------------------------
    # Reset environment
    # ------------------------------------------------------

    state = env.reset()

    if max_steps is None:
        max_steps = env.num_nodes - 1

    # ------------------------------------------------------
    # Storage
    # ------------------------------------------------------

    states = []
    actions = []
    rewards = []
    log_probs = []
    values = []
    entropies = []

    # ------------------------------------------------------
    # Routing history for LSTM
    # ------------------------------------------------------

    state_history = []

    # ------------------------------------------------------
    # Episode loop
    # ------------------------------------------------------

    for step in range(max_steps):

        # --------------------------------------------------
        # Dynamic node representation
        # --------------------------------------------------

        dynamic_node_state = dynamic_encoder(
            node_features=node_features,
            travel_features=travel_features,
            visited=state["visited"],
            current_node=state["current_node"],
            adjacency=adjacency
        )

        # --------------------------------------------------
        # Aggregate node states
        # --------------------------------------------------

        route_state = dynamic_node_state.mean(
            dim=0,
            keepdim=True
        )

        # --------------------------------------------------
        # Store route state in temporal history
        # --------------------------------------------------

        state_history.append(
            route_state.squeeze(0)
        )

        # Convert history into sequence
        sequence = torch.stack(
            state_history
        ).unsqueeze(0)

        # --------------------------------------------------
        # LSTM
        # --------------------------------------------------

        temporal_state = temporal_encoder(
            sequence
        )

        # --------------------------------------------------
        # Action mask
        # --------------------------------------------------

        action_mask = torch.tensor(
            1.0 - state["visited"],
            dtype=torch.float32,
            device=temporal_state.device
        ).unsqueeze(0)

        # --------------------------------------------------
        # PPO Actor-Critic
        # --------------------------------------------------

        logits, value = ppo_model(
            temporal_state,
            action_mask
        )

        # --------------------------------------------------
        # Probability distribution
        # --------------------------------------------------

        distribution = torch.distributions.Categorical(
            logits=logits
        )

        # Sample action
        action = distribution.sample()

        log_prob = distribution.log_prob(
            action
        )

        entropy = distribution.entropy()

        # --------------------------------------------------
        # Execute action
        # --------------------------------------------------

        action_idx = action.item()

        next_state, reward, done = env.step(
            action_idx
        )

        # --------------------------------------------------
        # Store transition
        # --------------------------------------------------

        states.append(
            temporal_state.squeeze(0).detach()
        )

        actions.append(
            action.detach()
        )

        rewards.append(
            float(reward)
        )

        log_probs.append(
            log_prob.detach()
        )

        values.append(
            value.squeeze().detach()
        )

        entropies.append(
            entropy.detach()
        )

        # --------------------------------------------------
        # Move to next state
        # --------------------------------------------------

        state = next_state

        if done:
            break

    # ------------------------------------------------------
    # Return trajectory
    # ------------------------------------------------------

    trajectory = {

        "states": torch.stack(states),

        "actions": torch.stack(actions),

        "rewards": torch.tensor(
            rewards,
            dtype=torch.float32
        ),

        "log_probs": torch.stack(
            log_probs
        ),

        "values": torch.stack(
            values
        ),

        "entropies": torch.stack(
            entropies
        ),

        "route": env.history.copy(),

        "total_time": env.total_time,

        "steps": len(rewards),

        "completed": env.deliveries_completed
    }

    return trajectory


print("Trajectory collector created successfully.")

Trajectory collector created successfully.


In [111]:
# ==========================================================
# CELL 30 — Test PPO Trajectory
# ==========================================================

trajectory = collect_trajectory(
    env=env,
    ppo_model=ppo_model,
    dynamic_encoder=dynamic_encoder,
    temporal_encoder=temporal_encoder,
    node_features=X,
    travel_features=travel_time_state,
    adjacency=A_normalized,
    max_steps=10
)

print("PPO TRAJECTORY")
print("=" * 60)

print(
    "Number of steps:",
    trajectory["steps"]
)

print(
    "Completed deliveries:",
    trajectory["completed"]
)

print(
    "Total travel time:",
    trajectory["total_time"],
    "seconds"
)

print(
    "State tensor shape:",
    trajectory["states"].shape
)

print(
    "Action tensor shape:",
    trajectory["actions"].shape
)

print(
    "Reward tensor shape:",
    trajectory["rewards"].shape
)

print(
    "Log-probability shape:",
    trajectory["log_probs"].shape
)

print(
    "Value tensor shape:",
    trajectory["values"].shape
)

print("\nGenerated route:")

for i, node_idx in enumerate(
    trajectory["route"]
):

    print(
        f"{i}: {env.idx_to_node[node_idx]}"
    )

PPO TRAJECTORY
Number of steps: 10
Completed deliveries: 10
Total travel time: 2286.4000396728516 seconds
State tensor shape: torch.Size([10, 32])
Action tensor shape: torch.Size([10, 1])
Reward tensor shape: torch.Size([10])
Log-probability shape: torch.Size([10, 1])
Value tensor shape: torch.Size([10])

Generated route:
0: UZ
1: WF
2: OR
3: RY
4: BY
5: HM
6: FY
7: MD
8: QA
9: HV
10: GL


In [112]:
# ==========================================================
# CELL 31 — PPO Returns and Advantages
# ==========================================================

def calculate_gae(
    rewards,
    values,
    gamma=0.99,
    gae_lambda=0.95,
    last_value=0.0
):

    # Make sure everything is 1-D
    rewards = rewards.flatten()
    values = values.flatten()

    advantages = torch.zeros_like(
        rewards
    )

    gae = 0.0

    # ------------------------------------------------------
    # Calculate GAE backwards
    # ------------------------------------------------------

    for t in reversed(
        range(len(rewards))
    ):

        if t == len(rewards) - 1:
            next_value = last_value
        else:
            next_value = values[t + 1]

        delta = (
            rewards[t]
            + gamma * next_value
            - values[t]
        )

        gae = (
            delta
            + gamma
            * gae_lambda
            * gae
        )

        advantages[t] = gae

    # ------------------------------------------------------
    # Returns
    # ------------------------------------------------------

    returns = (
        advantages
        + values
    )

    # ------------------------------------------------------
    # Normalize advantages
    # ------------------------------------------------------

    advantages = (
        advantages
        - advantages.mean()
    ) / (
        advantages.std() + 1e-8
    )

    return returns, advantages


# ==========================================================
# CALCULATE FOR OUR TEST TRAJECTORY
# ==========================================================

returns, advantages = calculate_gae(
    rewards=trajectory["rewards"],
    values=trajectory["values"]
)

print("PPO GAE")
print("=" * 60)

print(
    "Returns shape:",
    returns.shape
)

print(
    "Advantages shape:",
    advantages.shape
)

print(
    "Return mean:",
    returns.mean().item()
)

print(
    "Return std:",
    returns.std().item()
)

print(
    "Advantage mean:",
    advantages.mean().item()
)

print(
    "Advantage std:",
    advantages.std().item()
)

print("\nFirst 5 returns:")
print(returns[:5])

print("\nFirst 5 advantages:")
print(advantages[:5])

PPO GAE
Returns shape: torch.Size([10])
Advantages shape: torch.Size([10])
Return mean: -847.2424926757812
Return std: 514.7491455078125
Advantage mean: -6.556511067401516e-08
Advantage std: 1.0

First 5 returns:
tensor([-1895.0706, -1320.7528, -1032.0575,  -907.5544,  -911.9105])

First 5 advantages:
tensor([-2.0356, -0.9199, -0.3590, -0.1172, -0.1256])


In [113]:
# ==========================================================
# CELL 32 — PPO Loss and Optimizer
# ==========================================================

optimizer = torch.optim.Adam(
    ppo_model.parameters(),
    lr=3e-4
)

PPO_CLIP = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.01


def ppo_update(
    model,
    optimizer,
    states,
    actions,
    old_log_probs,
    returns,
    advantages
):

    # ------------------------------------------------------
    # Fix tensor dimensions
    # ------------------------------------------------------

    states = states.detach()

    actions = actions.flatten().detach()

    old_log_probs = (
        old_log_probs
        .flatten()
        .detach()
    )

    returns = returns.flatten().detach()

    advantages = advantages.flatten().detach()

    # ------------------------------------------------------
    # PPO update
    # ------------------------------------------------------

    logits, values = model(
        states
    )

    values = values.squeeze(-1)

    distribution = torch.distributions.Categorical(
        logits=logits
    )

    new_log_probs = distribution.log_prob(
        actions
    )

    entropy = distribution.entropy().mean()

    # ------------------------------------------------------
    # Probability ratio
    # ------------------------------------------------------

    ratio = torch.exp(
        new_log_probs - old_log_probs
    )

    # ------------------------------------------------------
    # Clipped surrogate objective
    # ------------------------------------------------------

    unclipped = (
        ratio * advantages
    )

    clipped = (
        torch.clamp(
            ratio,
            1.0 - PPO_CLIP,
            1.0 + PPO_CLIP
        )
        * advantages
    )

    actor_loss = -torch.min(
        unclipped,
        clipped
    ).mean()

    # ------------------------------------------------------
    # Critic loss
    # ------------------------------------------------------

    critic_loss = torch.nn.functional.mse_loss(
        values,
        returns
    )

    # ------------------------------------------------------
    # Total PPO loss
    # ------------------------------------------------------

    total_loss = (
        actor_loss
        + VALUE_COEF * critic_loss
        - ENTROPY_COEF * entropy
    )

    # ------------------------------------------------------
    # Backpropagation
    # ------------------------------------------------------

    optimizer.zero_grad()

    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=0.5
    )

    optimizer.step()

    return {
        "total_loss": total_loss.item(),
        "actor_loss": actor_loss.item(),
        "critic_loss": critic_loss.item(),
        "entropy": entropy.item(),
        "ratio_mean": ratio.mean().item()
    }


print("PPO optimizer created successfully.")
print("Learning rate:", 3e-4)
print("PPO clip:", PPO_CLIP)
print("Value coefficient:", VALUE_COEF)
print("Entropy coefficient:", ENTROPY_COEF)

PPO optimizer created successfully.
Learning rate: 0.0003
PPO clip: 0.2
Value coefficient: 0.5
Entropy coefficient: 0.01


In [114]:
# ==========================================================
# CELL 33 — Masked PPO Update
# ==========================================================

def ppo_update(
    model,
    optimizer,
    states,
    actions,
    old_log_probs,
    returns,
    advantages,
    action_masks
):

    # ------------------------------------------------------
    # Flatten dimensions
    # ------------------------------------------------------

    states = states.detach()

    actions = actions.flatten().detach()

    old_log_probs = (
        old_log_probs
        .flatten()
        .detach()
    )

    returns = (
        returns
        .flatten()
        .detach()
    )

    advantages = (
        advantages
        .flatten()
        .detach()
    )

    # ------------------------------------------------------
    # Actor-Critic forward pass
    # ------------------------------------------------------

    logits, values = model(
        states,
        action_masks
    )

    values = values.squeeze(-1)

    # ------------------------------------------------------
    # Probability distribution
    # ------------------------------------------------------

    distribution = torch.distributions.Categorical(
        logits=logits
    )

    new_log_probs = distribution.log_prob(
        actions
    )

    entropy = distribution.entropy().mean()

    # ------------------------------------------------------
    # PPO probability ratio
    # ------------------------------------------------------

    ratio = torch.exp(
        new_log_probs
        - old_log_probs
    )

    # ------------------------------------------------------
    # Clipped objective
    # ------------------------------------------------------

    unclipped = (
        ratio
        * advantages
    )

    clipped = (
        torch.clamp(
            ratio,
            1.0 - PPO_CLIP,
            1.0 + PPO_CLIP
        )
        * advantages
    )

    actor_loss = -torch.min(
        unclipped,
        clipped
    ).mean()

    # ------------------------------------------------------
    # Critic loss
    # ------------------------------------------------------

    critic_loss = torch.nn.functional.mse_loss(
        values,
        returns
    )

    # ------------------------------------------------------
    # Total PPO loss
    # ------------------------------------------------------

    total_loss = (
        actor_loss
        + VALUE_COEF * critic_loss
        - ENTROPY_COEF * entropy
    )

    # ------------------------------------------------------
    # Backpropagation
    # ------------------------------------------------------

    optimizer.zero_grad()

    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=0.5
    )

    optimizer.step()

    return {
        "total_loss": total_loss.item(),
        "actor_loss": actor_loss.item(),
        "critic_loss": critic_loss.item(),
        "entropy": entropy.item(),
        "ratio_mean": ratio.mean().item()
    }


print("Masked PPO update function created successfully.")

Masked PPO update function created successfully.


In [115]:
# ==========================================================
# CELL 34 — PPO Trajectory Collector with Action Masks
# ==========================================================

def collect_trajectory(
    env,
    ppo_model,
    dynamic_encoder,
    temporal_encoder,
    node_features,
    travel_features,
    adjacency,
    max_steps=None
):

    # ------------------------------------------------------
    # Reset environment
    # ------------------------------------------------------

    state = env.reset()

    if max_steps is None:
        max_steps = env.num_nodes - 1

    # ------------------------------------------------------
    # Storage
    # ------------------------------------------------------

    states = []
    actions = []
    rewards = []
    log_probs = []
    values = []
    entropies = []
    action_masks = []

    # ------------------------------------------------------
    # LSTM routing history
    # ------------------------------------------------------

    state_history = []

    # ------------------------------------------------------
    # Episode
    # ------------------------------------------------------

    for step in range(max_steps):

        # --------------------------------------------------
        # Dynamic node state
        # --------------------------------------------------

        dynamic_node_state = dynamic_encoder(
            node_features=node_features,
            travel_features=travel_features,
            visited=state["visited"],
            current_node=state["current_node"],
            adjacency=adjacency
        )

        # --------------------------------------------------
        # Route-level state
        # --------------------------------------------------

        route_state = dynamic_node_state.mean(
            dim=0,
            keepdim=True
        )

        # --------------------------------------------------
        # Add to temporal history
        # --------------------------------------------------

        state_history.append(
            route_state.squeeze(0)
        )

        sequence = torch.stack(
            state_history
        ).unsqueeze(0)

        # --------------------------------------------------
        # LSTM
        # --------------------------------------------------

        temporal_state = temporal_encoder(
            sequence
        )

        # --------------------------------------------------
        # Action mask
        # --------------------------------------------------

        action_mask = torch.tensor(
            1.0 - state["visited"],
            dtype=torch.float32,
            device=temporal_state.device
        ).unsqueeze(0)

        # --------------------------------------------------
        # PPO Actor-Critic
        # --------------------------------------------------

        logits, value = ppo_model(
            temporal_state,
            action_mask
        )

        # --------------------------------------------------
        # Distribution
        # --------------------------------------------------

        distribution = torch.distributions.Categorical(
            logits=logits
        )

        action = distribution.sample()

        log_prob = distribution.log_prob(
            action
        )

        entropy = distribution.entropy()

        # --------------------------------------------------
        # Execute action
        # --------------------------------------------------

        action_idx = action.item()

        next_state, reward, done = env.step(
            action_idx
        )

        # --------------------------------------------------
        # Store transition
        # --------------------------------------------------

        states.append(
            temporal_state.squeeze(0).detach()
        )

        actions.append(
            action.detach()
        )

        rewards.append(
            float(reward)
        )

        log_probs.append(
            log_prob.detach()
        )

        values.append(
            value.squeeze().detach()
        )

        entropies.append(
            entropy.detach()
        )

        action_masks.append(
            action_mask.squeeze(0).detach()
        )

        # --------------------------------------------------
        # Next environment state
        # --------------------------------------------------

        state = next_state

        if done:
            break

    # ------------------------------------------------------
    # Convert trajectory to tensors
    # ------------------------------------------------------

    trajectory = {

        "states":
            torch.stack(states),

        "actions":
            torch.stack(actions),

        "rewards":
            torch.tensor(
                rewards,
                dtype=torch.float32
            ),

        "log_probs":
            torch.stack(log_probs),

        "values":
            torch.stack(values),

        "entropies":
            torch.stack(entropies),

        "action_masks":
            torch.stack(action_masks),

        "route":
            env.history.copy(),

        "total_time":
            env.total_time,

        "steps":
            len(rewards),

        "completed":
            env.deliveries_completed
    }

    return trajectory


print(
    "Updated trajectory collector created successfully."
)

Updated trajectory collector created successfully.


In [116]:
# ==========================================================
# CELL 35 — Test Updated Trajectory
# ==========================================================

trajectory = collect_trajectory(
    env=env,
    ppo_model=ppo_model,
    dynamic_encoder=dynamic_encoder,
    temporal_encoder=temporal_encoder,
    node_features=X,
    travel_features=travel_time_state,
    adjacency=A_normalized,
    max_steps=10
)

print("UPDATED PPO TRAJECTORY")
print("=" * 60)

print(
    "Steps:",
    trajectory["steps"]
)

print(
    "Completed:",
    trajectory["completed"]
)

print(
    "Total travel time:",
    trajectory["total_time"],
    "seconds"
)

print("\nTensor shapes:")

print(
    "States:",
    trajectory["states"].shape
)

print(
    "Actions:",
    trajectory["actions"].shape
)

print(
    "Rewards:",
    trajectory["rewards"].shape
)

print(
    "Log probabilities:",
    trajectory["log_probs"].shape
)

print(
    "Values:",
    trajectory["values"].shape
)

print(
    "Action masks:",
    trajectory["action_masks"].shape
)

print(
    "\nAvailable actions at first step:",
    int(
        trajectory["action_masks"][0].sum().item()
    )
)

print(
    "Available actions at last step:",
    int(
        trajectory["action_masks"][-1].sum().item()
    )
)

UPDATED PPO TRAJECTORY
Steps: 10
Completed: 10
Total travel time: 3003.1999702453613 seconds

Tensor shapes:
States: torch.Size([10, 32])
Actions: torch.Size([10, 1])
Rewards: torch.Size([10])
Log probabilities: torch.Size([10, 1])
Values: torch.Size([10])
Action masks: torch.Size([10, 166])

Available actions at first step: 165
Available actions at last step: 156


In [117]:
# ==========================================================
# CELL 36 — GAE for Updated Trajectory
# ==========================================================

returns, advantages = calculate_gae(
    rewards=trajectory["rewards"],
    values=trajectory["values"]
)

print("PPO GAE — UPDATED TRAJECTORY")
print("=" * 60)

print("Returns shape:", returns.shape)
print("Advantages shape:", advantages.shape)

print("\nReturn statistics")
print("Mean:", returns.mean().item())
print("Std :", returns.std().item())
print("Min :", returns.min().item())
print("Max :", returns.max().item())

print("\nAdvantage statistics")
print("Mean:", advantages.mean().item())
print("Std :", advantages.std().item())
print("Min :", advantages.min().item())
print("Max :", advantages.max().item())

print("\nFirst 5 returns:")
print(returns[:5])

print("\nFirst 5 advantages:")
print(advantages[:5])

PPO GAE — UPDATED TRAJECTORY
Returns shape: torch.Size([10])
Advantages shape: torch.Size([10])

Return statistics
Mean: -1133.603759765625
Std : 672.5574340820312
Min : -2476.05419921875
Max : -289.1000061035156

Advantage statistics
Mean: 4.7683716530855236e-08
Std : 1.0
Min : -1.9960403442382812
Max : 1.2556599378585815

First 5 returns:
tensor([-2476.0542, -1801.7561, -1570.8170, -1338.3459, -1139.3337])

First 5 advantages:
tensor([-1.9960, -0.9934, -0.6501, -0.3044, -0.0085])


In [118]:
# ==========================================================
# CELL 37 — FIRST PPO GRADIENT UPDATE
# ==========================================================

update_info = ppo_update(
    model=ppo_model,
    optimizer=optimizer,

    states=trajectory["states"],

    actions=trajectory["actions"],

    old_log_probs=trajectory["log_probs"],

    returns=returns,

    advantages=advantages,

    action_masks=trajectory["action_masks"]
)

print("FIRST PPO UPDATE")
print("=" * 60)

print("Total loss :", update_info["total_loss"])
print("Actor loss :", update_info["actor_loss"])
print("Critic loss:", update_info["critic_loss"])
print("Entropy    :", update_info["entropy"])
print("Ratio mean :", update_info["ratio_mean"])

FIRST PPO UPDATE
Total loss : 846020.625
Actor loss : -0.0
Critic loss: 1692041.375
Entropy    : 5.076573371887207
Ratio mean : 1.0


In [119]:
# ==========================================================
# CELL 38 — FULL MODEL OPTIMIZER
# ==========================================================

# ----------------------------------------------------------
# Train ALL neural components
# ----------------------------------------------------------

trainable_parameters = list(
    dynamic_encoder.parameters()
) + list(
    temporal_encoder.parameters()
) + list(
    ppo_model.parameters()
)

optimizer = torch.optim.Adam(
    trainable_parameters,
    lr=3e-4
)

print("Full model optimizer created.")
print("=" * 60)

print(
    "Dynamic encoder parameters:",
    sum(
        p.numel()
        for p in dynamic_encoder.parameters()
    )
)

print(
    "Temporal encoder parameters:",
    sum(
        p.numel()
        for p in temporal_encoder.parameters()
    )
)

print(
    "PPO parameters:",
    sum(
        p.numel()
        for p in ppo_model.parameters()
    )
)

print(
    "Total trainable parameters:",
    sum(
        p.numel()
        for p in trainable_parameters
    )
)

Full model optimizer created.
Dynamic encoder parameters: 7264
Temporal encoder parameters: 7168
PPO parameters: 42279
Total trainable parameters: 56711


In [120]:
# ==========================================================
# CELL 39 — END-TO-END GRADIENT FLOW TEST
# ==========================================================

print("Testing gradient flow through:")
print("Dynamic GCN -> LSTM -> PPO Actor/Critic")
print("=" * 60)

# ----------------------------------------------------------
# Reset environment
# ----------------------------------------------------------

test_state = env.reset()

# ----------------------------------------------------------
# Dynamic node representation
# ----------------------------------------------------------

dynamic_node_state = dynamic_encoder(
    node_features=X,
    travel_features=travel_time_state,
    visited=test_state["visited"],
    current_node=test_state["current_node"],
    adjacency=A_normalized
)

print("Dynamic node state:")
print(dynamic_node_state.shape)

# ----------------------------------------------------------
# Route-level representation
# ----------------------------------------------------------

route_state = dynamic_node_state.mean(
    dim=0,
    keepdim=True
)

print("Route state:")
print(route_state.shape)

# ----------------------------------------------------------
# LSTM temporal encoding
# ----------------------------------------------------------

lstm_input = route_state.unsqueeze(1)

temporal_state = temporal_encoder(
    lstm_input
)

print("Temporal state:")
print(temporal_state.shape)

# ----------------------------------------------------------
# Action mask
# ----------------------------------------------------------

action_mask = torch.tensor(
    1.0 - test_state["visited"],
    dtype=torch.float32,
    device=temporal_state.device
).unsqueeze(0)

print("Action mask:")
print(action_mask.shape)

# ----------------------------------------------------------
# PPO Actor + Critic
# ----------------------------------------------------------

logits, value = ppo_model(
    temporal_state,
    action_mask
)

print("Actor logits:")
print(logits.shape)

print("Critic value:")
print(value.shape)

# ----------------------------------------------------------
# Create a policy distribution
# ----------------------------------------------------------

distribution = torch.distributions.Categorical(
    logits=logits
)

# Select the highest-probability valid action
test_action = torch.argmax(logits, dim=-1)

log_prob = distribution.log_prob(
    test_action
)

# ----------------------------------------------------------
# Test loss
# ----------------------------------------------------------

# Simple policy + value test objective.
# This is ONLY a gradient-flow test.
policy_loss = -log_prob.mean()

value_target = torch.tensor(
    [[-100.0]],
    dtype=value.dtype,
    device=value.device
)

value_loss = torch.nn.functional.mse_loss(
    value,
    value_target
)

test_loss = (
    policy_loss
    + 0.5 * value_loss
)

# ----------------------------------------------------------
# Clear gradients
# ----------------------------------------------------------

dynamic_encoder.zero_grad()
temporal_encoder.zero_grad()
ppo_model.zero_grad()

# ----------------------------------------------------------
# Backpropagation
# ----------------------------------------------------------

test_loss.backward()

# ----------------------------------------------------------
# Gradient statistics
# ----------------------------------------------------------

def gradient_statistics(model):

    total_norm = 0.0
    parameters_with_grad = 0

    for parameter in model.parameters():

        if parameter.grad is not None:

            parameters_with_grad += 1

            total_norm += (
                parameter.grad.detach()
                .norm()
                .item()
                ** 2
            )

    total_norm = total_norm ** 0.5

    return parameters_with_grad, total_norm


gcn_grad_count, gcn_grad_norm = gradient_statistics(
    dynamic_encoder
)

lstm_grad_count, lstm_grad_norm = gradient_statistics(
    temporal_encoder
)

ppo_grad_count, ppo_grad_norm = gradient_statistics(
    ppo_model
)

# ----------------------------------------------------------
# Results
# ----------------------------------------------------------

print("\nGradient Flow Results")
print("=" * 60)

print(
    "Dynamic encoder parameters with gradients:",
    gcn_grad_count
)

print(
    "Dynamic encoder gradient norm:",
    gcn_grad_norm
)

print(
    "Temporal encoder parameters with gradients:",
    lstm_grad_count
)

print(
    "Temporal encoder gradient norm:",
    lstm_grad_norm
)

print(
    "PPO parameters with gradients:",
    ppo_grad_count
)

print(
    "PPO gradient norm:",
    ppo_grad_norm
)

print("\nTest loss:", test_loss.item())

print("\nContains NaN:")
print(
    torch.isnan(test_loss).item()
)

print("Contains Inf:")
print(
    torch.isinf(test_loss).item()
)

print("\nEND-TO-END GRADIENT TEST COMPLETE")

Testing gradient flow through:
Dynamic GCN -> LSTM -> PPO Actor/Critic
Dynamic node state:
torch.Size([166, 22])
Route state:
torch.Size([1, 22])
Temporal state:
torch.Size([1, 32])
Action mask:
torch.Size([1, 166])
Actor logits:
torch.Size([1, 166])
Critic value:
torch.Size([1, 1])

Gradient Flow Results
Dynamic encoder parameters with gradients: 4
Dynamic encoder gradient norm: 0.6276060990553403
Temporal encoder parameters with gradients: 4
Temporal encoder gradient norm: 3.7856199897519334
PPO parameters with gradients: 8
PPO gradient norm: 123.1503433146244

Test loss: 4999.31005859375

Contains NaN:
False
Contains Inf:
False

END-TO-END GRADIENT TEST COMPLETE


In [121]:
# ==========================================================
# CELL 40 — END-TO-END PPO UPDATE
# ==========================================================

def ppo_update_end_to_end(
    model,
    optimizer,
    states,
    actions,
    old_log_probs,
    returns,
    advantages,
    action_masks,
    epochs=4,
    clip_epsilon=0.2,
    value_coefficient=0.5,
    entropy_coefficient=0.01,
    max_grad_norm=0.5
):

    # ------------------------------------------------------
    # Ensure tensors have expected dimensions
    # ------------------------------------------------------

    if actions.dim() > 1:
        actions = actions.squeeze(-1)

    if old_log_probs.dim() > 1:
        old_log_probs = old_log_probs.squeeze(-1)

    if returns.dim() > 1:
        returns = returns.squeeze(-1)

    if advantages.dim() > 1:
        advantages = advantages.squeeze(-1)

    # ------------------------------------------------------
    # Normalize advantages
    # ------------------------------------------------------

    advantages = (
        advantages - advantages.mean()
    ) / (
        advantages.std() + 1e-8
    )

    # ------------------------------------------------------
    # Statistics
    # ------------------------------------------------------

    epoch_results = []

    # ------------------------------------------------------
    # PPO optimization epochs
    # ------------------------------------------------------

    for epoch in range(epochs):

        # --------------------------------------------------
        # Forward pass
        #
        # states already represent the temporal state.
        # --------------------------------------------------

        logits, values = model(
            states,
            action_masks
        )

        # --------------------------------------------------
        # Mask invalid actions
        # --------------------------------------------------

        masked_logits = logits.masked_fill(
            action_masks <= 0,
            -1e9
        )

        distribution = torch.distributions.Categorical(
            logits=masked_logits
        )

        # --------------------------------------------------
        # Current policy probabilities
        # --------------------------------------------------

        new_log_probs = distribution.log_prob(
            actions
        )

        entropy = distribution.entropy().mean()

        # --------------------------------------------------
        # PPO probability ratio
        # --------------------------------------------------

        ratio = torch.exp(
            new_log_probs - old_log_probs
        )

        # --------------------------------------------------
        # Clipped surrogate objective
        # --------------------------------------------------

        unclipped_objective = (
            ratio * advantages
        )

        clipped_ratio = torch.clamp(
            ratio,
            1.0 - clip_epsilon,
            1.0 + clip_epsilon
        )

        clipped_objective = (
            clipped_ratio * advantages
        )

        actor_loss = -torch.min(
            unclipped_objective,
            clipped_objective
        ).mean()

        # --------------------------------------------------
        # Critic loss
        # --------------------------------------------------

        values = values.squeeze(-1)

        critic_loss = torch.nn.functional.mse_loss(
            values,
            returns
        )

        # --------------------------------------------------
        # Entropy regularization
        # --------------------------------------------------

        total_loss = (
            actor_loss
            + value_coefficient * critic_loss
            - entropy_coefficient * entropy
        )

        # --------------------------------------------------
        # Clear gradients
        # --------------------------------------------------

        optimizer.zero_grad()

        # --------------------------------------------------
        # Backpropagation
        # --------------------------------------------------

        total_loss.backward()

        # --------------------------------------------------
        # Gradient clipping
        # --------------------------------------------------

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_grad_norm
        )

        # --------------------------------------------------
        # Optimizer step
        # --------------------------------------------------

        optimizer.step()

        # --------------------------------------------------
        # Store statistics
        # --------------------------------------------------

        epoch_results.append({

            "total_loss":
                total_loss.item(),

            "actor_loss":
                actor_loss.item(),

            "critic_loss":
                critic_loss.item(),

            "entropy":
                entropy.item(),

            "ratio_mean":
                ratio.mean().item(),

            "ratio_min":
                ratio.min().item(),

            "ratio_max":
                ratio.max().item(),

            "gradient_norm":
                float(gradient_norm)
        })

    # ------------------------------------------------------
    # Average statistics
    # ------------------------------------------------------

    result = {

        "total_loss":
            sum(
                x["total_loss"]
                for x in epoch_results
            ) / epochs,

        "actor_loss":
            sum(
                x["actor_loss"]
                for x in epoch_results
            ) / epochs,

        "critic_loss":
            sum(
                x["critic_loss"]
                for x in epoch_results
            ) / epochs,

        "entropy":
            sum(
                x["entropy"]
                for x in epoch_results
            ) / epochs,

        "ratio_mean":
            sum(
                x["ratio_mean"]
                for x in epoch_results
            ) / epochs,

        "ratio_min":
            min(
                x["ratio_min"]
                for x in epoch_results
            ),

        "ratio_max":
            max(
                x["ratio_max"]
                for x in epoch_results
            ),

        "gradient_norm":
            sum(
                x["gradient_norm"]
                for x in epoch_results
            ) / epochs,

        "epochs":
            epochs
    }

    return result


print("End-to-end PPO update function created successfully.")
print("=" * 60)
print("PPO epochs: 4")
print("Clip epsilon:", 0.2)
print("Value coefficient:", 0.5)
print("Entropy coefficient:", 0.01)
print("Gradient clipping:", 0.5)

End-to-end PPO update function created successfully.
PPO epochs: 4
Clip epsilon: 0.2
Value coefficient: 0.5
Entropy coefficient: 0.01
Gradient clipping: 0.5


In [122]:
# ==========================================================
# CELL 41 — TEST END-TO-END PPO UPDATE
# ==========================================================

print("TESTING END-TO-END PPO UPDATE")
print("=" * 60)

# ----------------------------------------------------------
# Verify trajectory tensors
# ----------------------------------------------------------

states = trajectory["states"]
actions = trajectory["actions"]
old_log_probs = trajectory["log_probs"]
action_masks = trajectory["action_masks"]

print("States:", states.shape)
print("Actions:", actions.shape)
print("Old log probabilities:", old_log_probs.shape)
print("Action masks:", action_masks.shape)
print("Returns:", returns.shape)
print("Advantages:", advantages.shape)

# ----------------------------------------------------------
# IMPORTANT:
# Use the full model optimizer created in Cell 38.
# ----------------------------------------------------------

update_result = ppo_update_end_to_end(
    model=ppo_model,
    optimizer=optimizer,
    states=states,
    actions=actions,
    old_log_probs=old_log_probs,
    returns=returns,
    advantages=advantages,
    action_masks=action_masks,
    epochs=4
)

# ----------------------------------------------------------
# Display results
# ----------------------------------------------------------

print("\nPPO UPDATE RESULTS")
print("=" * 60)

print(
    "Average total loss:",
    update_result["total_loss"]
)

print(
    "Average actor loss:",
    update_result["actor_loss"]
)

print(
    "Average critic loss:",
    update_result["critic_loss"]
)

print(
    "Average entropy:",
    update_result["entropy"]
)

print(
    "Average probability ratio:",
    update_result["ratio_mean"]
)

print(
    "Minimum probability ratio:",
    update_result["ratio_min"]
)

print(
    "Maximum probability ratio:",
    update_result["ratio_max"]
)

print(
    "Average gradient norm:",
    update_result["gradient_norm"]
)

print(
    "PPO epochs:",
    update_result["epochs"]
)

# ----------------------------------------------------------
# Safety checks
# ----------------------------------------------------------

values_to_check = [
    update_result["total_loss"],
    update_result["actor_loss"],
    update_result["critic_loss"],
    update_result["entropy"],
    update_result["ratio_mean"],
    update_result["gradient_norm"]
]

contains_nan = any(
    not torch.isfinite(
        torch.tensor(value)
    )
    for value in values_to_check
)

print("\nSafety check")
print("=" * 60)
print("Contains NaN/Inf:", contains_nan)

if not contains_nan:
    print("✓ PPO update completed successfully.")
else:
    print("✗ Numerical instability detected.")

TESTING END-TO-END PPO UPDATE
States: torch.Size([10, 32])
Actions: torch.Size([10, 1])
Old log probabilities: torch.Size([10, 1])
Action masks: torch.Size([10, 166])
Returns: torch.Size([10])
Advantages: torch.Size([10])

PPO UPDATE RESULTS
Average total loss: 845999.0625
Average actor loss: -0.0018863349978346378
Average critic loss: 1691998.25
Average entropy: 5.076549291610718
Average probability ratio: 1.0026241838932037
Minimum probability ratio: 0.9969272017478943
Maximum probability ratio: 1.011763572692871
Average gradient norm: 1404.9057006835938
PPO epochs: 4

Safety check
Contains NaN/Inf: False
✓ PPO update completed successfully.


In [123]:
# ==========================================================
# CELL 42 — REWARD SCALING
# ==========================================================

REWARD_SCALE = 100

# Original travel-time rewards
original_rewards = trajectory["rewards"].clone().detach()

# Scale rewards for PPO training
scaled_rewards = original_rewards / REWARD_SCALE

print("REWARD SCALING")
print("=" * 60)

print("Reward scale:", REWARD_SCALE)

print("\nOriginal rewards")
print("Mean:", original_rewards.mean().item())
print("Min :", original_rewards.min().item())
print("Max :", original_rewards.max().item())

print("\nScaled rewards")
print("Mean:", scaled_rewards.mean().item())
print("Min :", scaled_rewards.min().item())
print("Max :", scaled_rewards.max().item())

print("\nFirst 5 original rewards:")
print(original_rewards[:5])

print("\nFirst 5 scaled rewards:")
print(scaled_rewards[:5])

print("\nActual travel times are NOT changed.")
print("Only PPO training rewards are scaled.")

REWARD SCALING
Reward scale: 100

Original rewards
Mean: -300.3199768066406
Min : -781.5
Max : -26.700000762939453

Scaled rewards
Mean: -3.003200054168701
Min : -7.815000057220459
Max : -0.2670000195503235

First 5 original rewards:
tensor([-781.5000, -324.4000, -312.1000, -266.8000, -338.4000])

First 5 scaled rewards:
tensor([-7.8150, -3.2440, -3.1210, -2.6680, -3.3840])

Actual travel times are NOT changed.
Only PPO training rewards are scaled.


In [124]:
# ==========================================================
# CELL 43 — GAE WITH SCALED REWARDS
# ==========================================================

GAMMA = 0.99
GAE_LAMBDA = 0.95

# ----------------------------------------------------------
# GAE FUNCTION
# ----------------------------------------------------------

def compute_gae(
    rewards,
    values,
    gamma=0.99,
    gae_lambda=0.95
):
    """
    Compute Generalized Advantage Estimation (GAE).

    rewards:
        Tensor of shape [T]

    values:
        Tensor of shape [T]

    Returns:
        returns     -> discounted value targets
        advantages  -> GAE advantages
    """

    rewards = rewards.float()
    values = values.float()

    T = rewards.size(0)

    advantages = torch.zeros_like(rewards)

    gae = torch.tensor(
        0.0,
        dtype=rewards.dtype,
        device=rewards.device
    )

    for t in reversed(range(T)):

        # No future state after the final trajectory step.
        if t == T - 1:
            next_value = torch.tensor(
                0.0,
                dtype=values.dtype,
                device=values.device
            )
        else:
            next_value = values[t + 1]

        delta = (
            rewards[t]
            + gamma * next_value
            - values[t]
        )

        gae = (
            delta
            + gamma
            * gae_lambda
            * gae
        )

        advantages[t] = gae

    returns = advantages + values

    return returns, advantages


# ----------------------------------------------------------
# USE SCALED REWARDS
# ----------------------------------------------------------

rewards_for_training = (
    trajectory["rewards"] / REWARD_SCALE
)

# ----------------------------------------------------------
# CURRENT VALUE ESTIMATES
# ----------------------------------------------------------

with torch.no_grad():

    _, current_values = ppo_model(
        trajectory["states"],
        trajectory["action_masks"]
    )

    current_values = current_values.squeeze(-1)


# ----------------------------------------------------------
# COMPUTE GAE
# ----------------------------------------------------------

returns_scaled, advantages_scaled = compute_gae(
    rewards=rewards_for_training,
    values=current_values,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA
)


# ----------------------------------------------------------
# NORMALIZE ADVANTAGES
# ----------------------------------------------------------

advantages_scaled = (
    advantages_scaled
    - advantages_scaled.mean()
) / (
    advantages_scaled.std() + 1e-8
)


# ----------------------------------------------------------
# DISPLAY RESULTS
# ----------------------------------------------------------

print("SCALED-REWARD PPO GAE")
print("=" * 60)

print("Rewards shape:", rewards_for_training.shape)
print("Returns shape:", returns_scaled.shape)
print("Advantages shape:", advantages_scaled.shape)


print("\nScaled reward statistics")
print(
    "Mean:",
    rewards_for_training.mean().item()
)

print(
    "Std :",
    rewards_for_training.std().item()
)

print(
    "Min :",
    rewards_for_training.min().item()
)

print(
    "Max :",
    rewards_for_training.max().item()
)


print("\nReturn statistics")
print(
    "Mean:",
    returns_scaled.mean().item()
)

print(
    "Std :",
    returns_scaled.std().item()
)

print(
    "Min :",
    returns_scaled.min().item()
)

print(
    "Max :",
    returns_scaled.max().item()
)


print("\nAdvantage statistics")
print(
    "Mean:",
    advantages_scaled.mean().item()
)

print(
    "Std :",
    advantages_scaled.std().item()
)

print(
    "Min :",
    advantages_scaled.min().item()
)

print(
    "Max :",
    advantages_scaled.max().item()
)


print("\nFirst 5 scaled returns:")
print(returns_scaled[:5])


print("\nFirst 5 normalized advantages:")
print(advantages_scaled[:5])


# ----------------------------------------------------------
# NUMERICAL SAFETY CHECK
# ----------------------------------------------------------

has_nan = (
    torch.isnan(returns_scaled).any()
    or torch.isnan(advantages_scaled).any()
)

has_inf = (
    torch.isinf(returns_scaled).any()
    or torch.isinf(advantages_scaled).any()
)

print("\nSafety check")
print("=" * 60)

print("Contains NaN:", has_nan.item())
print("Contains Inf:", has_inf.item())

if not has_nan and not has_inf:
    print("✓ Scaled GAE calculated successfully.")
else:
    print("✗ Numerical instability detected.")

SCALED-REWARD PPO GAE
Rewards shape: torch.Size([10])
Returns shape: torch.Size([10])
Advantages shape: torch.Size([10])

Scaled reward statistics
Mean: -3.003200054168701
Std : 1.9415969848632812
Min : -7.815000057220459
Max : -0.2670000195503235

Return statistics
Mean: -11.353200912475586
Std : 6.735653877258301
Min : -24.79216957092285
Max : -2.8910000324249268

Advantage statistics
Mean: -1.3113022134803032e-07
Std : 1.0
Min : -1.9954999685287476
Max : 1.2561267614364624

First 5 scaled returns:
tensor([-24.7922, -18.0466, -15.7343, -13.4065, -11.4131])

First 5 normalized advantages:
tensor([-1.9955, -0.9937, -0.6502, -0.3046, -0.0087])

Safety check
Contains NaN: False
Contains Inf: False
✓ Scaled GAE calculated successfully.


In [125]:
# ==========================================================
# CELL 44 — PPO UPDATE WITH SCALED RETURNS
# ==========================================================

print("PPO UPDATE — SCALED REWARDS")
print("=" * 60)

scaled_update_result = ppo_update_end_to_end(
    model=ppo_model,
    optimizer=optimizer,
    states=trajectory["states"],
    actions=trajectory["actions"],
    old_log_probs=trajectory["log_probs"],
    returns=returns_scaled,
    advantages=advantages_scaled,
    action_masks=trajectory["action_masks"],
    epochs=4,
    clip_epsilon=0.2,
    value_coefficient=0.5,
    entropy_coefficient=0.01,
    max_grad_norm=0.5
)

print("\nPPO UPDATE RESULTS")
print("=" * 60)

print(
    "Average total loss:",
    scaled_update_result["total_loss"]
)

print(
    "Average actor loss:",
    scaled_update_result["actor_loss"]
)

print(
    "Average critic loss:",
    scaled_update_result["critic_loss"]
)

print(
    "Average entropy:",
    scaled_update_result["entropy"]
)

print(
    "Average probability ratio:",
    scaled_update_result["ratio_mean"]
)

print(
    "Minimum probability ratio:",
    scaled_update_result["ratio_min"]
)

print(
    "Maximum probability ratio:",
    scaled_update_result["ratio_max"]
)

print(
    "Average gradient norm:",
    scaled_update_result["gradient_norm"]
)

print(
    "PPO epochs:",
    scaled_update_result["epochs"]
)

# ----------------------------------------------------------
# Safety check
# ----------------------------------------------------------

check_values = [
    scaled_update_result["total_loss"],
    scaled_update_result["actor_loss"],
    scaled_update_result["critic_loss"],
    scaled_update_result["entropy"],
    scaled_update_result["ratio_mean"],
    scaled_update_result["gradient_norm"]
]

contains_invalid = any(
    not torch.isfinite(
        torch.tensor(value)
    )
    for value in check_values
)

print("\nSafety check")
print("=" * 60)

print(
    "Contains NaN/Inf:",
    contains_invalid
)

if not contains_invalid:
    print("✓ Scaled PPO update completed successfully.")
else:
    print("✗ Numerical instability detected.")

PPO UPDATE — SCALED REWARDS

PPO UPDATE RESULTS
Average total loss: 83.68889236450195
Average actor loss: -0.0038095489144325256
Average critic loss: 167.4869384765625
Average entropy: 5.076503038406372
Average probability ratio: 1.0064820647239685
Minimum probability ratio: 0.9960244297981262
Maximum probability ratio: 1.022843837738037
Average gradient norm: 13.893702030181885
PPO epochs: 4

Safety check
Contains NaN/Inf: False
✓ Scaled PPO update completed successfully.


In [126]:
# ==========================================================
# CELL 45 — FULL 165-DELIVERY PPO ROUTE
# ==========================================================

print("FULL ROUTE-LEVEL PPO EPISODE")
print("=" * 60)

print("Route ID:", ROUTE_ID)
print("Warehouse:", warehouse_id)
print("Total nodes:", len(node_ids))
print("Expected deliveries:", len(node_ids) - 1)

# ----------------------------------------------------------
# Generate a COMPLETE route
# ----------------------------------------------------------

trajectory = collect_trajectory(
    env=env,
    ppo_model=ppo_model,
    dynamic_encoder=dynamic_encoder,
    temporal_encoder=temporal_encoder,
    node_features=X,
    travel_features=travel_time_state,
    adjacency=A_normalized,
    max_steps=len(node_ids) - 1
)

# ----------------------------------------------------------
# Results
# ----------------------------------------------------------

print("\nFULL ROUTE RESULTS")
print("=" * 60)

print(
    "Steps:",
    trajectory["steps"]
)

print(
    "Completed deliveries:",
    trajectory["completed"]
)

print(
    "Total travel time:",
    trajectory["total_time"],
    "seconds"
)

print(
    "Total travel time:",
    trajectory["total_time"] / 60.0,
    "minutes"
)

# ----------------------------------------------------------
# Tensor shapes
# ----------------------------------------------------------

print("\nTrajectory tensor shapes")
print("=" * 60)

print(
    "States:",
    trajectory["states"].shape
)

print(
    "Actions:",
    trajectory["actions"].shape
)

print(
    "Rewards:",
    trajectory["rewards"].shape
)

print(
    "Log probabilities:",
    trajectory["log_probs"].shape
)

print(
    "Values:",
    trajectory["values"].shape
)

print(
    "Action masks:",
    trajectory["action_masks"].shape
)

# ----------------------------------------------------------
# Generated route
# ----------------------------------------------------------

print("\nGENERATED ROUTE")
print("=" * 60)

generated_route_ids = [
    idx_to_node[node_idx]
    for node_idx in trajectory["route"]
]

print(
    "Number of route nodes:",
    len(generated_route_ids)
)

print(
    "First 15 nodes:"
)

for i, node_id in enumerate(
    generated_route_ids[:15]
):
    print(
        f"{i}: {node_id}"
    )

print("\nLast 10 nodes:")

start = max(
    0,
    len(generated_route_ids) - 10
)

for i in range(
    start,
    len(generated_route_ids)
):
    print(
        f"{i}: {generated_route_ids[i]}"
    )

# ----------------------------------------------------------
# Route validity
# ----------------------------------------------------------

unique_nodes = len(
    set(generated_route_ids)
)

expected_nodes = len(node_ids)

print("\nROUTE VALIDITY")
print("=" * 60)

print(
    "Expected nodes:",
    expected_nodes
)

print(
    "Generated nodes:",
    len(generated_route_ids)
)

print(
    "Unique nodes:",
    unique_nodes
)

print(
    "All nodes visited:",
    unique_nodes == expected_nodes
)

print(
    "All deliveries completed:",
    trajectory["completed"] == expected_nodes - 1
)

print(
    "Episode finished:",
    trajectory["completed"] == expected_nodes - 1
)

FULL ROUTE-LEVEL PPO EPISODE
Route ID: RouteID_00092558-dece-4fb7-8d0d-7d0df3a4864e
Warehouse: UZ
Total nodes: 166
Expected deliveries: 165

FULL ROUTE RESULTS
Steps: 165
Completed deliveries: 165
Total travel time: 39662.099904060364 seconds
Total travel time: 661.0349984010061 minutes

Trajectory tensor shapes
States: torch.Size([165, 32])
Actions: torch.Size([165, 1])
Rewards: torch.Size([165])
Log probabilities: torch.Size([165, 1])
Values: torch.Size([165])
Action masks: torch.Size([165, 166])

GENERATED ROUTE
Number of route nodes: 166
First 15 nodes:
0: UZ
1: BB
2: BY
3: WR
4: CW
5: NJ
6: XT
7: YR
8: UM
9: JM
10: AJ
11: UY
12: VJ
13: ST
14: TZ

Last 10 nodes:
156: HJ
157: QV
158: PF
159: OZ
160: SG
161: GL
162: HT
163: YA
164: YY
165: AN

ROUTE VALIDITY
Expected nodes: 166
Generated nodes: 166
Unique nodes: 166
All nodes visited: True
All deliveries completed: True
Episode finished: True


In [127]:
# ==========================================================
# CELL 46 — FULL-ROUTE PPO TRAINING UPDATE
# ==========================================================

print("FULL-ROUTE PPO TRAINING UPDATE")
print("=" * 60)

# ----------------------------------------------------------
# Reward scaling
# ----------------------------------------------------------

REWARD_SCALE = 100.0
rewards_for_training = (
    trajectory["rewards"] / REWARD_SCALE
)

print("Number of steps:", len(rewards_for_training))
print(
    "Original reward mean:",
    trajectory["rewards"].mean().item()
)
print(
    "Scaled reward mean:",
    rewards_for_training.mean().item()
)

# ----------------------------------------------------------
# Current value estimates
# ----------------------------------------------------------

current_values = trajectory["values"].detach()

# ----------------------------------------------------------
# Compute GAE
# ----------------------------------------------------------

returns_scaled, advantages_scaled = compute_gae(
    rewards=rewards_for_training,
    values=current_values,
    gamma=0.99,
    gae_lambda=0.95
)

# ----------------------------------------------------------
# PPO update
# ----------------------------------------------------------

full_route_update = ppo_update_end_to_end(
    model=ppo_model,
    optimizer=optimizer,

    states=trajectory["states"],
    actions=trajectory["actions"],

    old_log_probs=trajectory["log_probs"],

    returns=returns_scaled,
    advantages=advantages_scaled,

    action_masks=trajectory["action_masks"],

    epochs=4,
    clip_epsilon=0.2,
    value_coefficient=0.5,
    entropy_coefficient=0.01,
    max_grad_norm=0.5
)

# ----------------------------------------------------------
# Results
# ----------------------------------------------------------

print("\nFULL-ROUTE PPO UPDATE RESULTS")
print("=" * 60)

print(
    "Route travel time:",
    trajectory["total_time"],
    "seconds"
)

print(
    "Route travel time:",
    trajectory["total_time"] / 60.0,
    "minutes"
)

print(
    "Completed deliveries:",
    trajectory["completed"]
)

print(
    "PPO total loss:",
    full_route_update["total_loss"]
)

print(
    "Actor loss:",
    full_route_update["actor_loss"]
)

print(
    "Critic loss:",
    full_route_update["critic_loss"]
)

print(
    "Entropy:",
    full_route_update["entropy"]
)

print(
    "Probability ratio:",
    full_route_update["ratio_mean"]
)

print(
    "Gradient norm:",
    full_route_update["gradient_norm"]
)

print(
    "PPO epochs:",
    full_route_update["epochs"]
)

# ----------------------------------------------------------
# Safety check
# ----------------------------------------------------------

check_values = [
    full_route_update["total_loss"],
    full_route_update["actor_loss"],
    full_route_update["critic_loss"],
    full_route_update["entropy"],
    full_route_update["ratio_mean"],
    full_route_update["gradient_norm"]
]

contains_invalid = any(
    not torch.isfinite(
        torch.tensor(value)
    )
    for value in check_values
)

print("\nSafety check")
print("=" * 60)

print(
    "Contains NaN/Inf:",
    contains_invalid
)

if not contains_invalid:
    print("✓ Full-route PPO update completed successfully.")
else:
    print("✗ Numerical instability detected.")

FULL-ROUTE PPO TRAINING UPDATE
Number of steps: 165
Original reward mean: -240.37637329101562
Scaled reward mean: -2.40376353263855

FULL-ROUTE PPO UPDATE RESULTS
Route travel time: 39662.099904060364 seconds
Route travel time: 661.0349984010061 minutes
Completed deliveries: 165
PPO total loss: 694.7576904296875
Actor loss: -0.0005016549148209037
Critic loss: 1389.598876953125
Entropy: 4.125493884086609
Probability ratio: 0.9992485195398331
Gradient norm: 45.51527500152588
PPO epochs: 4

Safety check
Contains NaN/Inf: False
✓ Full-route PPO update completed successfully.


In [128]:
# ==========================================================
# CELL 47 — MULTI-EPISODE PPO TRAINING
# ==========================================================

print("MULTI-EPISODE PPO TRAINING")
print("=" * 60)

# ----------------------------------------------------------
# Training configuration
# ----------------------------------------------------------

NUM_EPISODES = 20

GAMMA = 0.99
GAE_LAMBDA = 0.95

PPO_EPOCHS = 4
CLIP_EPSILON = 0.2
VALUE_COEFFICIENT = 0.5
ENTROPY_COEFFICIENT = 0.01
MAX_GRAD_NORM = 0.5

REWARD_SCALE = 1000.0

# ----------------------------------------------------------
# Training history
# ----------------------------------------------------------

training_history = []

best_travel_time = float("inf")
best_route = None
best_episode = None

# ----------------------------------------------------------
# Training loop
# ----------------------------------------------------------

for episode in range(1, NUM_EPISODES + 1):

    # ------------------------------------------------------
    # Generate a complete route
    # ------------------------------------------------------

    trajectory = collect_trajectory(
        env=env,
        ppo_model=ppo_model,
        dynamic_encoder=dynamic_encoder,
        temporal_encoder=temporal_encoder,
        node_features=X,
        travel_features=travel_time_state,
        adjacency=A_normalized,
        max_steps=len(node_ids) - 1
    )

    # ------------------------------------------------------
    # Scale rewards for PPO
    # ------------------------------------------------------

    rewards_for_training = (
        trajectory["rewards"] / REWARD_SCALE
    )

    # ------------------------------------------------------
    # Value estimates
    # ------------------------------------------------------

    current_values = trajectory["values"].detach()

    # ------------------------------------------------------
    # GAE
    # ------------------------------------------------------

    returns_scaled, advantages_scaled = compute_gae(
        rewards=rewards_for_training,
        values=current_values,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA
    )

    # ------------------------------------------------------
    # PPO update
    # ------------------------------------------------------

    update_result = ppo_update_end_to_end(
        model=ppo_model,
        optimizer=optimizer,

        states=trajectory["states"],
        actions=trajectory["actions"],

        old_log_probs=trajectory["log_probs"],

        returns=returns_scaled,
        advantages=advantages_scaled,

        action_masks=trajectory["action_masks"],

        epochs=PPO_EPOCHS,
        clip_epsilon=CLIP_EPSILON,
        value_coefficient=VALUE_COEFFICIENT,
        entropy_coefficient=ENTROPY_COEFFICIENT,
        max_grad_norm=MAX_GRAD_NORM
    )

    # ------------------------------------------------------
    # Record results
    # ------------------------------------------------------

    travel_time = trajectory["total_time"]

    completed = trajectory["completed"]

    training_history.append({
        "episode": episode,
        "travel_time": travel_time,
        "completed": completed,
        "total_loss": update_result["total_loss"],
        "actor_loss": update_result["actor_loss"],
        "critic_loss": update_result["critic_loss"],
        "entropy": update_result["entropy"],
        "ratio": update_result["ratio_mean"],
        "gradient_norm": update_result["gradient_norm"]
    })

    # ------------------------------------------------------
    # Best route
    # ------------------------------------------------------

    if (
        completed == len(node_ids) - 1
        and travel_time < best_travel_time
    ):
        best_travel_time = travel_time
        best_route = trajectory["route"].copy()
        best_episode = episode

    # ------------------------------------------------------
    # Episode output
    # ------------------------------------------------------

    print(
        f"Episode {episode:02d}/{NUM_EPISODES} | "
        f"Steps: {trajectory['steps']:3d} | "
        f"Completed: {completed:3d} | "
        f"Time: {travel_time:10.2f}s | "
        f"Loss: {update_result['total_loss']:.4f} | "
        f"Entropy: {update_result['entropy']:.4f}"
    )

# ----------------------------------------------------------
# Training summary
# ----------------------------------------------------------

print("\n")
print("TRAINING COMPLETE")
print("=" * 60)

print(
    "Episodes completed:",
    len(training_history)
)

print(
    "Best episode:",
    best_episode
)

print(
    "Best travel time:",
    best_travel_time,
    "seconds"
)

print(
    "Best travel time:",
    best_travel_time / 60.0,
    "minutes"
)

print(
    "Best route completed:",
    best_route is not None
)

# ----------------------------------------------------------
# Improvement from first episode
# ----------------------------------------------------------

if len(training_history) > 0:

    first_time = training_history[0]["travel_time"]

    improvement = (
        (first_time - best_travel_time)
        / first_time
    ) * 100.0

    print(
        "Improvement from first episode:",
        improvement,
        "%"
    )

# ----------------------------------------------------------
# Numerical safety
# ----------------------------------------------------------

invalid_training_values = False

for record in training_history:

    for key in [
        "travel_time",
        "total_loss",
        "actor_loss",
        "critic_loss",
        "entropy",
        "ratio",
        "gradient_norm"
    ]:

        if not torch.isfinite(
            torch.tensor(record[key])
        ):
            invalid_training_values = True

            break

    if invalid_training_values:
        break

print("\nSafety check")
print("=" * 60)

print(
    "Contains NaN/Inf:",
    invalid_training_values
)

if not invalid_training_values:
    print(
        "✓ Multi-episode PPO training completed successfully."
    )
else:
    print(
        "✗ Numerical instability detected."
    )

MULTI-EPISODE PPO TRAINING
Episode 01/20 | Steps: 165 | Completed: 165 | Time:   40624.40s | Loss: 7.2561 | Entropy: 4.1253
Episode 02/20 | Steps: 165 | Completed: 165 | Time:   41456.10s | Loss: 7.2265 | Entropy: 4.1251
Episode 03/20 | Steps: 165 | Completed: 165 | Time:   42437.10s | Loss: 7.4499 | Entropy: 4.1249
Episode 04/20 | Steps: 165 | Completed: 165 | Time:   43987.90s | Loss: 8.0468 | Entropy: 4.1248
Episode 05/20 | Steps: 165 | Completed: 165 | Time:   39499.80s | Loss: 6.4050 | Entropy: 4.1242
Episode 06/20 | Steps: 165 | Completed: 165 | Time:   41409.00s | Loss: 7.0706 | Entropy: 4.1240
Episode 07/20 | Steps: 165 | Completed: 165 | Time:   42951.20s | Loss: 7.3509 | Entropy: 4.1239
Episode 08/20 | Steps: 165 | Completed: 165 | Time:   41582.30s | Loss: 7.2742 | Entropy: 4.1236
Episode 09/20 | Steps: 165 | Completed: 165 | Time:   40960.50s | Loss: 6.8033 | Entropy: 4.1225
Episode 10/20 | Steps: 165 | Completed: 165 | Time:   41962.60s | Loss: 7.1362 | Entropy: 4.1229
Epi

In [129]:
# ==========================================================
# CELL 48 — EXTENDED PPO TRAINING
# ==========================================================

print("EXTENDED PPO TRAINING")
print("=" * 60)

NUM_EPISODES = 500

GAMMA = 0.99
GAE_LAMBDA = 0.95

PPO_EPOCHS = 4
CLIP_EPSILON = 0.2
VALUE_COEFFICIENT = 0.5
ENTROPY_COEFFICIENT = 0.01
MAX_GRAD_NORM = 0.5

REWARD_SCALE = 1000.0

extended_history = []

best_travel_time = float("inf")
best_route = None
best_episode = None

for episode in range(1, NUM_EPISODES + 1):

    # ------------------------------------------------------
    # Generate complete 165-delivery route
    # ------------------------------------------------------

    trajectory = collect_trajectory(
        env=env,
        ppo_model=ppo_model,
        dynamic_encoder=dynamic_encoder,
        temporal_encoder=temporal_encoder,
        node_features=X,
        travel_features=travel_time_state,
        adjacency=A_normalized,
        max_steps=len(node_ids) - 1
    )

    # ------------------------------------------------------
    # Scaled rewards
    # ------------------------------------------------------

    rewards_for_training = (
        trajectory["rewards"] / REWARD_SCALE
    )

    # ------------------------------------------------------
    # GAE
    # ------------------------------------------------------

    current_values = trajectory["values"].detach()

    returns_scaled, advantages_scaled = compute_gae(
        rewards=rewards_for_training,
        values=current_values,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA
    )

    # ------------------------------------------------------
    # PPO update
    # ------------------------------------------------------

    update_result = ppo_update_end_to_end(
        model=ppo_model,
        optimizer=optimizer,
        states=trajectory["states"],
        actions=trajectory["actions"],
        old_log_probs=trajectory["log_probs"],
        returns=returns_scaled,
        advantages=advantages_scaled,
        action_masks=trajectory["action_masks"],
        epochs=PPO_EPOCHS,
        clip_epsilon=CLIP_EPSILON,
        value_coefficient=VALUE_COEFFICIENT,
        entropy_coefficient=ENTROPY_COEFFICIENT,
        max_grad_norm=MAX_GRAD_NORM
    )

    travel_time = trajectory["total_time"]
    completed = trajectory["completed"]

    extended_history.append({
        "episode": episode,
        "travel_time": travel_time,
        "completed": completed,
        "loss": update_result["total_loss"],
        "actor_loss": update_result["actor_loss"],
        "critic_loss": update_result["critic_loss"],
        "entropy": update_result["entropy"]
    })

    # ------------------------------------------------------
    # Best route
    # ------------------------------------------------------

    if (
        completed == len(node_ids) - 1
        and travel_time < best_travel_time
    ):
        best_travel_time = travel_time
        best_route = trajectory["route"].copy()
        best_episode = episode

    # ------------------------------------------------------
    # Progress
    # ------------------------------------------------------

    print(
        f"Episode {episode:03d}/{NUM_EPISODES} | "
        f"Time: {travel_time:10.2f}s | "
        f"Completed: {completed:3d} | "
        f"Loss: {update_result['total_loss']:.4f} | "
        f"Entropy: {update_result['entropy']:.4f}"
    )


# ----------------------------------------------------------
# Final result
# ----------------------------------------------------------

print("\nEXTENDED TRAINING COMPLETE")
print("=" * 60)

print("Episodes:", len(extended_history))

print(
    "Best episode:",
    best_episode
)

print(
    "Best travel time:",
    best_travel_time,
    "seconds"
)

print(
    "Best travel time:",
    best_travel_time / 60.0,
    "minutes"
)

print(
    "Best route completed:",
    best_route is not None
)

# ----------------------------------------------------------
# Improvement
# ----------------------------------------------------------

if len(extended_history) > 0:

    first_time = extended_history[0]["travel_time"]

    improvement = (
        (first_time - best_travel_time)
        / first_time
    ) * 100.0

    print(
        "Improvement from first episode:",
        improvement,
        "%"
    )

print("\nTraining safety check")
print("=" * 60)

invalid = False

for record in extended_history:

    for key in [
        "travel_time",
        "loss",
        "actor_loss",
        "critic_loss",
        "entropy"
    ]:

        if not torch.isfinite(
            torch.tensor(record[key])
        ):
            invalid = True
            break

    if invalid:
        break

print("Contains NaN/Inf:", invalid)

if not invalid:
    print("✓ Extended PPO training completed successfully.")
else:
    print("✗ Numerical instability detected.")

EXTENDED PPO TRAINING
Episode 001/500 | Time:   41145.20s | Completed: 165 | Loss: 6.1326 | Entropy: 4.1048
Episode 002/500 | Time:   39160.10s | Completed: 165 | Loss: 5.5598 | Entropy: 4.0989
Episode 003/500 | Time:   41966.50s | Completed: 165 | Loss: 6.2488 | Entropy: 4.0949
Episode 004/500 | Time:   38575.60s | Completed: 165 | Loss: 5.1477 | Entropy: 4.0924
Episode 005/500 | Time:   40915.40s | Completed: 165 | Loss: 5.8792 | Entropy: 4.0893
Episode 006/500 | Time:   41517.10s | Completed: 165 | Loss: 5.8517 | Entropy: 4.0832
Episode 007/500 | Time:   40391.60s | Completed: 165 | Loss: 5.7643 | Entropy: 4.0844
Episode 008/500 | Time:   42021.50s | Completed: 165 | Loss: 5.8269 | Entropy: 4.0776
Episode 009/500 | Time:   42055.90s | Completed: 165 | Loss: 5.8312 | Entropy: 4.0673
Episode 010/500 | Time:   41623.60s | Completed: 165 | Loss: 5.6730 | Entropy: 4.0723
Episode 011/500 | Time:   41192.90s | Completed: 165 | Loss: 5.3571 | Entropy: 4.0663
Episode 012/500 | Time:   42401.

In [130]:
# ==========================================================
# CELL 49A — VERIFY DYNAMIC ENCODER SIGNATURE
# ==========================================================

import inspect

print("DynamicStateEncoder forward signature")
print("=" * 60)

print(
    inspect.signature(
        dynamic_encoder.forward
    )
)

print("\ncollect_trajectory signature")
print("=" * 60)

print(
    inspect.signature(
        collect_trajectory
    )
)

DynamicStateEncoder forward signature
(node_features, travel_features, visited, current_node, adjacency)

collect_trajectory signature
(env, ppo_model, dynamic_encoder, temporal_encoder, node_features, travel_features, adjacency, max_steps=None)


In [131]:
# ==========================================================
# CELL 49 — DETERMINISTIC PPO EVALUATION — FINAL FIX
# ==========================================================

print("DETERMINISTIC PPO EVALUATION")
print("=" * 60)

ppo_model.eval()
dynamic_encoder.eval()
temporal_encoder.eval()

NUM_EVAL_RUNS = 10


def collect_deterministic_trajectory(
    env,
    ppo_model,
    dynamic_encoder,
    temporal_encoder,
    node_features,
    travel_features,
    adjacency,
    max_steps=None
):

    # ------------------------------------------------------
    # RESET
    # ------------------------------------------------------

    reset_result = env.reset()

    if isinstance(reset_result, tuple):
        observation = reset_result[0]
    else:
        observation = reset_result

    num_nodes = node_features.shape[0]

    if max_steps is None:
        max_steps = num_nodes - 1

    # ------------------------------------------------------
    # STORAGE
    # ------------------------------------------------------

    states = []
    actions = []
    rewards = []
    log_probs = []
    values = []
    action_masks = []

    route_indices = []

    # ------------------------------------------------------
    # INITIAL NODE
    # ------------------------------------------------------

    current_node = int(env.current_node)

    route_indices.append(current_node)

    # ------------------------------------------------------
    # VISITED STATE
    # ------------------------------------------------------

    visited = torch.zeros(
        num_nodes,
        dtype=torch.bool,
        device=node_features.device
    )

    visited[current_node] = True

    # ------------------------------------------------------
    # MAIN LOOP
    # ------------------------------------------------------

    for step in range(max_steps):

        # ==================================================
        # DYNAMIC GCN
        # ==================================================

        with torch.no_grad():

            dynamic_node_state = dynamic_encoder(
                node_features,
                travel_features,
                visited,
                current_node,
                adjacency
            )

        # ==================================================
        # ROUTE STATE
        # ==================================================

        route_state = dynamic_node_state.mean(
            dim=0,
            keepdim=True
        )

        # ==================================================
        # LSTM
        # ==================================================

        with torch.no_grad():

            temporal_state = temporal_encoder(
                route_state.unsqueeze(1)
            )

        # ==================================================
        # ACTION MASK
        #
        # True  = action available
        # False = action unavailable
        #
        # Visited nodes cannot be selected again.
        # ==================================================

        action_mask = (~visited).clone()

        # Current node is already visited, so it is masked.

        action_mask = action_mask.unsqueeze(0)

        # ==================================================
        # PPO FORWARD
        # ==================================================

        with torch.no_grad():

            logits, value = ppo_model(
                temporal_state,
                action_mask
            )

            masked_logits = logits.clone()

            masked_logits[
                ~action_mask
            ] = -1e9

            probabilities = torch.softmax(
                masked_logits,
                dim=-1
            )

            # ----------------------------------------------
            # DETERMINISTIC ACTION
            # ----------------------------------------------

            action = torch.argmax(
                probabilities,
                dim=-1
            )

            action_index = action.item()

            selected_probability = probabilities[
                0,
                action_index
            ]

            selected_log_prob = torch.log(
                selected_probability.clamp(
                    min=1e-8
                )
            )

        # ==================================================
        # STORE
        # ==================================================

        states.append(
            temporal_state.squeeze(0).detach()
        )

        actions.append(
            action.detach()
        )

        log_probs.append(
            selected_log_prob.detach()
        )

        values.append(
            value.squeeze().detach()
        )

        action_masks.append(
            action_mask.squeeze(0).detach()
        )

        # ==================================================
        # ENVIRONMENT STEP
        # ==================================================

        result = env.step(
            action_index
        )

        if isinstance(result, tuple):

            if len(result) == 5:

                (
                    next_observation,
                    reward,
                    terminated,
                    truncated,
                    info
                ) = result

                done = (
                    terminated
                    or truncated
                )

            elif len(result) == 4:

                (
                    next_observation,
                    reward,
                    done,
                    info
                ) = result

            else:

                next_observation = result[0]
                reward = result[1]
                done = result[2]
                info = {}

        else:

            next_observation = result
            reward = 0.0
            done = False
            info = {}

        # --------------------------------------------------
        # STORE ACTUAL TRAVEL-TIME REWARD
        # --------------------------------------------------

        rewards.append(
            float(reward)
        )

        # ==================================================
        # UPDATE CURRENT NODE
        # ==================================================

        current_node = int(
            env.current_node
        )

        route_indices.append(
            current_node
        )

        visited[current_node] = True

        observation = next_observation

        # --------------------------------------------------
        # FINISH
        # --------------------------------------------------

        if done:
            break

    # ======================================================
    # NODE NAMES
    # ======================================================

    route = [
        node_ids[idx]
        for idx in route_indices
    ]

    # ======================================================
    # TOTAL ROUTE TIME
    # ======================================================

    if hasattr(
        env,
        "total_route_time"
    ):

        total_time = float(
            env.total_route_time
        )

    elif hasattr(
        env,
        "total_time"
    ):

        total_time = float(
            env.total_time
        )

    else:

        # Rewards are negative travel times.
        total_time = float(
            -sum(rewards)
        )

    # ======================================================
    # COMPLETED DELIVERIES
    # ======================================================

    if hasattr(
        env,
        "deliveries_completed"
    ):

        completed = int(
            env.deliveries_completed
        )

    elif hasattr(
        env,
        "completed_deliveries"
    ):

        completed = int(
            env.completed_deliveries
        )

    else:

        completed = len(route) - 1

    # ======================================================
    # TENSORS
    # ======================================================

    states_tensor = torch.stack(
        states
    )

    actions_tensor = torch.stack(
        actions
    ).view(-1, 1)

    rewards_tensor = torch.tensor(
        rewards,
        dtype=torch.float32
    )

    log_probs_tensor = torch.stack(
        log_probs
    ).view(-1, 1)

    values_tensor = torch.stack(
        values
    )

    masks_tensor = torch.stack(
        action_masks
    )

    # ======================================================
    # RETURN
    # ======================================================

    return {

        "states":
            states_tensor,

        "actions":
            actions_tensor,

        "rewards":
            rewards_tensor,

        "log_probs":
            log_probs_tensor,

        "values":
            values_tensor,

        "action_masks":
            masks_tensor,

        "route":
            route,

        "route_indices":
            route_indices,

        "steps":
            len(rewards),

        "completed":
            completed,

        "total_time":
            total_time
    }


# ==========================================================
# RUN 10 DETERMINISTIC EVALUATIONS
# ==========================================================

evaluation_results = []

for run in range(
    1,
    NUM_EVAL_RUNS + 1
):

    trajectory = collect_deterministic_trajectory(

        env=env,

        ppo_model=ppo_model,

        dynamic_encoder=dynamic_encoder,

        temporal_encoder=temporal_encoder,

        node_features=X,

        travel_features=travel_time_state,

        adjacency=A_normalized,

        max_steps=len(node_ids) - 1
    )

    travel_time = trajectory[
        "total_time"
    ]

    completed = trajectory[
        "completed"
    ]

    route = trajectory[
        "route"
    ]

    unique_nodes = len(
        set(route)
    )

    valid = (
        len(route) == len(node_ids)
        and
        unique_nodes == len(node_ids)
        and
        completed == len(node_ids) - 1
    )

    evaluation_results.append({

        "run":
            run,

        "travel_time":
            travel_time,

        "completed":
            completed,

        "valid":
            valid,

        "route":
            route

    })

    print(
        f"Evaluation {run:02d}/{NUM_EVAL_RUNS} | "
        f"Steps: {trajectory['steps']:3d} | "
        f"Completed: {completed:3d} | "
        f"Time: {travel_time:10.2f}s | "
        f"Valid: {valid}"
    )


# ==========================================================
# SUMMARY
# ==========================================================

print()
print("EVALUATION SUMMARY")
print("=" * 60)

valid_results = [
    r
    for r in evaluation_results
    if r["valid"]
]

print(
    "Valid evaluations:",
    len(valid_results),
    "/",
    NUM_EVAL_RUNS
)

if len(valid_results) > 0:

    best_eval_result = min(
        valid_results,
        key=lambda x: x["travel_time"]
    )

    best_eval_time = (
        best_eval_result["travel_time"]
    )

    mean_eval_time = sum(
        r["travel_time"]
        for r in valid_results
    ) / len(valid_results)

    print(
        "Best deterministic time:",
        best_eval_time,
        "seconds"
    )

    print(
        "Best deterministic time:",
        best_eval_time / 60.0,
        "minutes"
    )

    print(
        "Mean deterministic time:",
        mean_eval_time,
        "seconds"
    )

    print(
        "Mean deterministic time:",
        mean_eval_time / 60.0,
        "minutes"
    )

    # ------------------------------------------------------
    # BEST ROUTE
    # ------------------------------------------------------

    print()
    print("BEST DETERMINISTIC ROUTE")
    print("=" * 60)

    for i, node in enumerate(
        best_eval_result["route"]
    ):

        print(
            f"{i}: {node}"
        )

else:

    print(
        "✗ No valid deterministic route generated."
    )


# ==========================================================
# TRAINING COMPARISON
# ==========================================================

print()
print("TRAINING vs DETERMINISTIC")
print("=" * 60)

print(
    "Best training route:",
    best_travel_time,
    "seconds"
)

print(
    "Best training route:",
    best_travel_time / 60.0,
    "minutes"
)

if len(valid_results) > 0:

    print(
        "Best deterministic route:",
        best_eval_time,
        "seconds"
    )

    difference = (
        (
            best_eval_time
            -
            best_travel_time
        )
        /
        best_travel_time
    ) * 100.0

    print(
        "Difference:",
        difference,
        "%"
    )


# ==========================================================
# SAFETY
# ==========================================================

invalid = False

for result in evaluation_results:

    if not torch.isfinite(
        torch.tensor(
            result["travel_time"]
        )
    ):

        invalid = True
        break

print()
print("Safety check")
print("=" * 60)

print(
    "Contains NaN/Inf:",
    invalid
)

if not invalid:

    print(
        "✓ Deterministic evaluation completed successfully."
    )

else:

    print(
        "✗ Numerical instability detected."
    )

DETERMINISTIC PPO EVALUATION
Evaluation 01/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True


/tmp/ipykernel_1586/2217363954.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  visited_tensor = torch.tensor(


Evaluation 02/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 03/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 04/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 05/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 06/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 07/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 08/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 09/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True
Evaluation 10/10 | Steps: 165 | Completed: 165 | Time:   36820.60s | Valid: True

EVALUATION SUMMARY
Valid evaluations: 10 / 10
Best deterministic time: 36820.60017776489 seconds
Best deterministic time: 613.6766696294148 minutes
Mean deterministic time: 36820.60017776489 seconds
Mean deterministic time: 613.6766696294148 minutes

BEST DETERMINISTIC 

In [132]:
# ==========================================================
# CELL 50A — FIND EXISTING ROUTING VARIABLES
# ==========================================================

print("AVAILABLE ROUTING VARIABLES")
print("=" * 60)

candidate_names = [
    "route_data",
    "travel_time_data",
    "travel_times",
    "travel_time_matrix",
    "travel_time_state",
    "node_ids",
    "selected_route_id",
    "selected_route_ids",
    "training_route_id",
    "route_id",
    "route",
    "X",
    "A",
    "adjacency",
    "A_normalized"
]

for name in candidate_names:

    if name in globals():

        obj = globals()[name]

        print(
            f"{name:25s} -> "
            f"{type(obj)}"
        )

        # Shape if available
        if hasattr(obj, "shape"):

            print(
                f"{'':25s}    shape = {obj.shape}"
            )

        # Length if available
        elif hasattr(obj, "__len__"):

            try:
                print(
                    f"{'':25s}    length = {len(obj)}"
                )
            except:
                pass

print()
print("ROUTE-RELATED DICTIONARY KEYS")
print("=" * 60)

if "route_data" in globals():

    print(
        "route_data keys:",
        list(route_data.keys())[:10]
    )

if "travel_time_data" in globals():

    print(
        "travel_time_data keys:",
        list(travel_time_data.keys())[:10]
    )

AVAILABLE ROUTING VARIABLES
route_data                -> <class 'dict'>
                             length = 3052
travel_time_matrix        -> <class 'numpy.ndarray'>
                             shape = (166, 166)
travel_time_state         -> <class 'torch.Tensor'>
                             shape = torch.Size([166, 4])
node_ids                  -> <class 'list'>
                             length = 166
route_id                  -> <class 'str'>
                             length = 44
route                     -> <class 'list'>
                             length = 166
X                         -> <class 'torch.Tensor'>
                             shape = torch.Size([166, 4])
A                         -> <class 'numpy.ndarray'>
                             shape = (166, 166)
A_normalized              -> <class 'torch.Tensor'>
                             shape = torch.Size([166, 166])

ROUTE-RELATED DICTIONARY KEYS
route_data keys: ['RouteID_00092558-dece-4fb7-8d0d-7d0df3a4864e'

In [133]:
# ==========================================================
# CELL 50 — NEAREST-NEIGHBOR BASELINE
# ==========================================================

print("NEAREST-NEIGHBOR BASELINE")
print("=" * 60)

# ----------------------------------------------------------
# USE EXISTING VARIABLES
# ----------------------------------------------------------

matrix = travel_time_matrix
nodes = node_ids

# Training route
baseline_route_id = ROUTE_ID

# ----------------------------------------------------------
# GET WAREHOUSE
# ----------------------------------------------------------

route_info = route_data[baseline_route_id]

print("Route ID:", baseline_route_id)

# Try to obtain warehouse from route information
if isinstance(route_info, dict):

    warehouse = route_info.get(
        "Warehouse",
        route_info.get(
            "warehouse",
            "UZ"
        )
    )

else:

    warehouse = "UZ"

print("Warehouse:", warehouse)

# ----------------------------------------------------------
# FIND STARTING NODE
# ----------------------------------------------------------

if warehouse in nodes:

    current_idx = nodes.index(warehouse)

else:

    print(
        "Warehouse not found in node_ids."
    )

    raise ValueError(
        f"Warehouse {warehouse} not found."
    )

# ----------------------------------------------------------
# INITIALIZE
# ----------------------------------------------------------

visited = set()

visited.add(current_idx)

baseline_route_indices = [
    current_idx
]

total_travel_time = 0.0

# ----------------------------------------------------------
# NEAREST-NEIGHBOR ROUTING
# ----------------------------------------------------------

for step in range(len(nodes) - 1):

    available = [
        idx
        for idx in range(len(nodes))
        if idx not in visited
    ]

    if len(available) == 0:
        break

    # Travel times from current node
    candidate_times = matrix[
        current_idx,
        available
    ]

    # Remove invalid values
    valid_candidates = []

    for i, travel_time in zip(
        available,
        candidate_times
    ):

        if np.isfinite(travel_time):

            valid_candidates.append(
                (
                    float(travel_time),
                    i
                )
            )

    if len(valid_candidates) == 0:

        print(
            "No valid destination found at step:",
            step
        )

        break

    # ------------------------------------------------------
    # SELECT CLOSEST UNVISITED NODE
    # ------------------------------------------------------

    travel_time, next_idx = min(
        valid_candidates,
        key=lambda x: x[0]
    )

    # ------------------------------------------------------
    # UPDATE
    # ------------------------------------------------------

    total_travel_time += travel_time

    current_idx = next_idx

    visited.add(
        current_idx
    )

    baseline_route_indices.append(
        current_idx
    )


# ==========================================================
# CONVERT INDICES TO NODE IDs
# ==========================================================

baseline_route = [
    nodes[idx]
    for idx in baseline_route_indices
]

# ==========================================================
# VALIDITY
# ==========================================================

unique_nodes = len(
    set(baseline_route_indices)
)

expected_nodes = len(nodes)

baseline_valid = (
    len(baseline_route_indices)
    == expected_nodes
    and
    unique_nodes
    == expected_nodes
)

# ==========================================================
# RESULTS
# ==========================================================

print()
print("BASELINE RESULTS")
print("=" * 60)

print(
    "Steps:",
    len(baseline_route) - 1
)

print(
    "Expected nodes:",
    expected_nodes
)

print(
    "Generated nodes:",
    len(baseline_route)
)

print(
    "Unique nodes:",
    unique_nodes
)

print(
    "All nodes visited:",
    baseline_valid
)

print(
    "Total travel time:",
    total_travel_time,
    "seconds"
)

print(
    "Total travel time:",
    total_travel_time / 60.0,
    "minutes"
)

# ==========================================================
# FIRST 15 NODES
# ==========================================================

print()
print("BASELINE ROUTE — FIRST 15 NODES")
print("=" * 60)

for i, node in enumerate(
    baseline_route[:15]
):

    print(
        f"{i}: {node}"
    )

# ==========================================================
# LAST 10 NODES
# ==========================================================

print()
print("BASELINE ROUTE — LAST 10 NODES")
print("=" * 60)

start = max(
    0,
    len(baseline_route) - 10
)

for i in range(
    start,
    len(baseline_route)
):

    print(
        f"{i}: {baseline_route[i]}"
    )

# ==========================================================
# PPO COMPARISON
# ==========================================================

ppo_time = 40578.599922180176

print()
print("BASELINE vs PPO")
print("=" * 60)

print(
    "Nearest-neighbor:",
    total_travel_time,
    "seconds"
)

print(
    "PPO deterministic:",
    ppo_time,
    "seconds"
)

if total_travel_time > 0:

    improvement = (
        (
            total_travel_time
            -
            ppo_time
        )
        /
        total_travel_time
    ) * 100.0

    print(
        "PPO improvement:",
        improvement,
        "%"
    )

# ==========================================================
# SAFETY CHECK
# ==========================================================

print()
print("Safety check")
print("=" * 60)

print(
    "Matrix shape:",
    matrix.shape
)

print(
    "Contains NaN/Inf:",
    not np.isfinite(matrix).all()
)

print(
    "Route valid:",
    baseline_valid
)

if baseline_valid:

    print(
        "✓ Nearest-neighbor baseline completed successfully."
    )

else:

    print(
        "✗ Baseline route is invalid."
    )

NEAREST-NEIGHBOR BASELINE
Route ID: RouteID_00092558-dece-4fb7-8d0d-7d0df3a4864e
Warehouse: UZ

BASELINE RESULTS
Steps: 165
Expected nodes: 166
Generated nodes: 166
Unique nodes: 166
All nodes visited: True
Total travel time: 8386.799992442131 seconds
Total travel time: 139.77999987403552 minutes

BASELINE ROUTE — FIRST 15 NODES
0: UZ
1: JT
2: OF
3: JM
4: EI
5: WC
6: FQ
7: EU
8: HJ
9: FC
10: QK
11: ER
12: LS
13: EC
14: HL

BASELINE ROUTE — LAST 10 NODES
156: QV
157: UY
158: XC
159: XL
160: HV
161: GL
162: AJ
163: RT
164: GQ
165: JN

BASELINE vs PPO
Nearest-neighbor: 8386.799992442131 seconds
PPO deterministic: 40578.599922180176 seconds
PPO improvement: -383.8388891919216 %

Safety check
Matrix shape: (166, 166)
Contains NaN/Inf: False
Route valid: True
✓ Nearest-neighbor baseline completed successfully.


In [134]:
# ==========================================================
# DIAGNOSTIC — FIND CURRENT TRAJECTORY VARIABLES
# ==========================================================

print("CURRENT PPO / TRAJECTORY VARIABLES")
print("=" * 60)

# Show likely trajectory-related variables
keywords = [
    "trajectory",
    "full",
    "ppo",
    "reward",
    "return",
    "advantage",
    "action"
]

found = []

for name in dir():

    name_lower = name.lower()

    if any(
        keyword in name_lower
        for keyword in keywords
    ):

        try:
            value = globals()[name]

            found.append(name)

            print(
                f"{name:35s} -> "
                f"{type(value)}"
            )

            if isinstance(value, dict):

                print(
                    " " * 4,
                    "keys:",
                    list(value.keys())
                )

            elif torch.is_tensor(value):

                print(
                    " " * 4,
                    "shape:",
                    tuple(value.shape)
                )

        except Exception:
            pass


print()
print("Total candidate variables:", len(found))

CURRENT PPO / TRAJECTORY VARIABLES
PPOActorCritic                      -> <class 'type'>
PPO_CLIP                            -> <class 'float'>
PPO_EPOCHS                          -> <class 'int'>
REWARD_SCALE                        -> <class 'float'>
action                              -> <class 'torch.Tensor'>
     shape: (1,)
action_idx                          -> <class 'int'>
action_mask                         -> <class 'torch.Tensor'>
     shape: (1, 166)
action_masks                        -> <class 'torch.Tensor'>
     shape: (10, 166)
actions                             -> <class 'torch.Tensor'>
     shape: (10, 1)
advantages                          -> <class 'torch.Tensor'>
     shape: (10,)
advantages_scaled                   -> <class 'torch.Tensor'>
     shape: (165,)
collect_deterministic_trajectory    -> <class 'function'>
collect_trajectory                  -> <class 'function'>
first_action                        -> <class 'numpy.int64'>
full_route_update            

In [135]:
# ==========================================================
# CHECK CURRENT ROUTING FUNCTIONS
# ==========================================================

import inspect

print("CURRENT ROUTING FUNCTIONS")
print("=" * 60)

for name in [
    "collect_deterministic_trajectory",
    "collect_trajectory",
    "ppo_model",
    "dynamic_encoder",
    "temporal_encoder"
]:

    if name in globals():

        obj = globals()[name]

        print(f"\n{name}")
        print("-" * 60)

        try:
            print(inspect.signature(obj))
        except Exception:
            print(type(obj))

    else:


        print(f"\n{name}: NOT FOUND")

CURRENT ROUTING FUNCTIONS

collect_deterministic_trajectory
------------------------------------------------------------
(env, ppo_model, dynamic_encoder, temporal_encoder, node_features, travel_features, adjacency, max_steps=None)

collect_trajectory
------------------------------------------------------------
(env, ppo_model, dynamic_encoder, temporal_encoder, node_features, travel_features, adjacency, max_steps=None)

ppo_model
------------------------------------------------------------
(*args, **kwargs)

dynamic_encoder
------------------------------------------------------------
(*args, **kwargs)

temporal_encoder
------------------------------------------------------------
(*args, **kwargs)


## Fallback Branch — TGCN + Transformer + SAC
Additive comparison branch. Original LSTM+PPO pipeline above is untouched.

### F1. Transformer Temporal Encoder (replaces LSTM)
Consumes full 166-node sequence per step — real multi-head attention, not attention-over-one-token.

In [136]:
# ==========================================================
# F1 — Transformer Temporal Encoder (replaces LSTM)
# ==========================================================

import math


class PositionalEncoding(nn.Module):

    def __init__(self, embed_dim, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2, dtype=torch.float32) *
            (-math.log(10000.0) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, embed_dim]

    def forward(self, x):
        # x: [batch, seq_len, embed_dim]
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :].to(x.device)


class TransformerTemporalEncoder(nn.Module):

    def __init__(
        self,
        input_size=22,
        embed_dim=32,
        num_heads=4,
        num_layers=2,
        ff_dim=128,
        dropout=0.1
    ):
        super().__init__()

        self.input_proj = nn.Linear(input_size, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.embed_dim = embed_dim

    def forward(self, x):
        # x: [batch, num_nodes, input_size]

        h = self.input_proj(x)
        h = self.pos_encoding(h)
        h = self.transformer(h)

        # Pool across node dimension -> route-level state
        pooled = h.mean(dim=1)

        return pooled  # [batch, embed_dim]


# ----------------------------------------------------------
# Create transformer temporal encoder (dynamic, 22-dim input)
# ----------------------------------------------------------

temporal_encoder_v2 = TransformerTemporalEncoder(
    input_size=22,
    embed_dim=32,
    num_heads=4,
    num_layers=2,
    ff_dim=128
)

print(temporal_encoder_v2)

TransformerTemporalEncoder(
  (input_proj): Linear(in_features=22, out_features=32, bias=True)
  (pos_encoding): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)


### F2. Generate Route State via Transformer
Reuses existing `dynamic_encoder` unchanged. Feeds full node sequence (no pre-pool) into transformer, pools after.

In [137]:
# ==========================================================
# F2 — Generate Route State via Transformer
# ==========================================================

state_v2 = env.reset()

dynamic_node_features_v2 = dynamic_encoder(
    node_features=X,
    travel_features=travel_time_state,
    visited=state_v2["visited"],
    current_node=state_v2["current_node"],
    adjacency=A_normalized
)

print("Dynamic node feature shape:")
print(dynamic_node_features_v2.shape)
print("Expected: (166, 22)")

# ----------------------------------------------------------
# Feed FULL node sequence into transformer (no mean-pool first)
# ----------------------------------------------------------

transformer_input_v2 = dynamic_node_features_v2.unsqueeze(0)  # [1, 166, 22]

print("\nTransformer input shape:")
print(transformer_input_v2.shape)

temporal_encoder_v2.eval()

with torch.no_grad():
    route_state_v2 = temporal_encoder_v2(transformer_input_v2)  # [1, 32]

print("\nRoute state shape:")
print(route_state_v2.shape)
print("Expected: torch.Size([1, 32])")

print("\nContains NaN:", torch.isnan(route_state_v2).any().item())
print("Contains Inf:", torch.isinf(route_state_v2).any().item())

Dynamic node feature shape:
torch.Size([166, 22])
Expected: (166, 22)

Transformer input shape:
torch.Size([1, 166, 22])

Route state shape:
torch.Size([1, 32])
Expected: torch.Size([1, 32])

Contains NaN: False
Contains Inf: False


In [138]:
# ==========================================================
# CELL 64 — SAC Replay Buffer
# ==========================================================
import random
from collections import deque

class SACReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action_mask, action, reward, next_state, next_action_mask, done):
        self.buffer.append((
            state.detach().squeeze(0),
            action_mask.detach().squeeze(0),
            action,
            reward,
            next_state.detach().squeeze(0),
            next_action_mask.detach().squeeze(0),
            done
        ))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, action_masks, actions, rewards, next_states, next_action_masks, dones = zip(*batch)

        return (
            torch.stack(states),
            torch.stack(action_masks),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.stack(next_states),
            torch.stack(next_action_masks),
            torch.tensor(dones, dtype=torch.float32)
        )

    def __len__(self):
        return len(self.buffer)

sac_buffer = SACReplayBuffer(capacity=100000)
print("SAC Replay Buffer initialized.")

SAC Replay Buffer initialized.


In [139]:
# ==========================================================
# CELL 65 — Discrete SAC Agent
# ==========================================================
class DiscreteSAC(nn.Module):
    def __init__(self, state_dim=32, num_actions=166, hidden_dim=128):
        super().__init__()

        # Critic: Twin Q-Networks
        self.q1 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions)
        )

        self.q2 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions)
        )

        # Target Networks
        self.target_q1 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions)
        )

        self.target_q2 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions)
        )

        self.target_q1.load_state_dict(self.q1.state_dict())
        self.target_q2.load_state_dict(self.q2.state_dict())

        # Actor Network
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions)
        )

    def get_action_probs(self, state, action_mask):
        logits = self.actor(state)

        logits = logits.masked_fill(action_mask == 0, -1e9)

        probs = torch.softmax(logits, dim=-1)
        log_probs = torch.log(probs + 1e-8)

        return probs, log_probs

    def select_action(self, state, action_mask, deterministic=False):

        probs, _ = self.get_action_probs(state, action_mask)

        if deterministic:
            return torch.argmax(probs, dim=-1).item()

        return torch.distributions.Categorical(probs).sample().item()


# ----------------------------------------------------------
# Create SAC model
# ----------------------------------------------------------

sac_model = DiscreteSAC(
    state_dim=32,
    num_actions=166,
    hidden_dim=128
)

device = next(dynamic_encoder.parameters()).device
sac_model = sac_model.to(device)

# ----------------------------------------------------------
# Optimizers
# ----------------------------------------------------------

ACTOR_LR = 3e-4
CRITIC_LR = 3e-4

actor_optimizer = torch.optim.Adam(
    sac_model.actor.parameters(),
    lr=ACTOR_LR
)

critic1_optimizer = torch.optim.Adam(
    sac_model.q1.parameters(),
    lr=CRITIC_LR
)

critic2_optimizer = torch.optim.Adam(
    sac_model.q2.parameters(),
    lr=CRITIC_LR
)

print(sac_model)

DiscreteSAC(
  (q1): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=166, bias=True)
  )
  (q2): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=166, bias=True)
  )
  (target_q1): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=166, bias=True)
  )
  (target_q2): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=166, bias=True)
  )
  (actor)

In [140]:
# ==========================================================
# CELL 66 — SAC Update Logic
# ==========================================================
def sac_update(batch_size, gamma=0.99, alpha=0.05, tau=0.005):

    if len(sac_buffer) < batch_size:
        return None

    states, action_masks, actions, rewards, next_states, next_action_masks, dones = sac_buffer.sample(batch_size)

    states = states.to(device)
    action_masks = action_masks.to(device)
    actions = actions.to(device).unsqueeze(1)
    rewards = rewards.to(device).unsqueeze(1)
    next_states = next_states.to(device)
    next_action_masks = next_action_masks.to(device)
    dones = dones.to(device).unsqueeze(1)

    # -----------------------------
    # Target Q
    # -----------------------------
    with torch.no_grad():

        next_probs, next_log_probs = sac_model.get_action_probs(
            next_states,
            next_action_masks
        )

        next_q1 = sac_model.target_q1(next_states)
        next_q2 = sac_model.target_q2(next_states)

        next_q = torch.min(next_q1, next_q2)

        next_v = (
            next_probs *
            (next_q - alpha * next_log_probs)
        ).sum(dim=1, keepdim=True)

        target_q = rewards + (1 - dones) * gamma * next_v

    # -----------------------------
    # Critic Update
    # -----------------------------

    current_q1 = sac_model.q1(states).gather(1, actions)
    current_q2 = sac_model.q2(states).gather(1, actions)

    critic_loss = (
        F.mse_loss(current_q1, target_q)
        +
        F.mse_loss(current_q2, target_q)
    )

    critic1_optimizer.zero_grad()
    critic2_optimizer.zero_grad()

    critic_loss.backward()

    critic1_optimizer.step()
    critic2_optimizer.step()

    # -----------------------------
    # Actor Update
    # -----------------------------

    probs, log_probs = sac_model.get_action_probs(
        states,
        action_masks
    )

    q1_pi = sac_model.q1(states)
    q2_pi = sac_model.q2(states)

    min_q = torch.min(q1_pi, q2_pi)

    actor_loss = (
        probs *
        (alpha * log_probs - min_q)
    ).sum(dim=1).mean()

    actor_optimizer.zero_grad()

    actor_loss.backward()

    actor_optimizer.step()

    # -----------------------------
    # Target Network Update
    # -----------------------------

    for target_param, param in zip(
        sac_model.target_q1.parameters(),
        sac_model.q1.parameters()
    ):
        target_param.data.copy_(
            tau * param.data +
            (1 - tau) * target_param.data
        )

    for target_param, param in zip(
        sac_model.target_q2.parameters(),
        sac_model.q2.parameters()
    ):
        target_param.data.copy_(
            tau * param.data +
            (1 - tau) * target_param.data
        )

    return (
        critic_loss.item(),
        actor_loss.item()
    )

print("SAC Update function initialized.")

SAC Update function initialized.


In [ ]:
# ==========================================================
# CELL 67 — Fallback Pipeline Training (TGCN + Transformer + SAC)
# ==========================================================
print("TRAINING TGCN + TRANSFORMER + SAC")
print("=" * 60)

SAC_EPISODES = 500
BATCH_SIZE = 64
REWARD_SCALE = 1000.0

sac_history = []
best_sac_time = float("inf")
best_sac_route = None

# Ensure encoders are in train mode
dynamic_encoder.train()
temporal_encoder_v2.train()

for episode in range(1, SAC_EPISODES + 1):
    state_dict = env.reset()
    
    current_node = state_dict["current_node"]
    visited_arr = state_dict["visited"]
    
    total_travel_time = 0
    episode_loss = []
    
    for step in range(len(node_ids) - 1):
        # 1. Spatial Embeddings via TGCN
        dynamic_node_state = dynamic_encoder(
            node_features=X.to(device),
            travel_features=travel_time_state.to(device),
            visited=visited_arr,
            current_node=current_node,
            adjacency=A_normalized.to(device)
        )
        
        # 2. Temporal State via Transformer
        transformer_input = dynamic_node_state.unsqueeze(0)
        route_state = temporal_encoder_v2(transformer_input) # [1, 32]
        
        # 3. Action Mask
        action_mask = torch.tensor(1.0 - visited_arr, dtype=torch.float32, device=device).unsqueeze(0)
        
        # 4. Action Selection
        action = sac_model.select_action(route_state, action_mask, deterministic=False)
        
        # 5. Environment Step
        next_state_dict, reward, done = env.step(action)
        total_travel_time -= reward  # Assuming reward is negative travel time
        
        # Next state representation for Buffer
        with torch.no_grad():
            next_dynamic = dynamic_encoder(
                node_features=X.to(device), travel_features=travel_time_state.to(device),
                visited=next_state_dict["visited"], current_node=next_state_dict["current_node"],
                adjacency=A_normalized.to(device)
            )
            next_route_state = temporal_encoder_v2(next_dynamic.unsqueeze(0))
            next_action_mask = torch.tensor(1.0 - next_state_dict["visited"], dtype=torch.float32, device=device).unsqueeze(0)
            
        # 6. Push to Buffer
        scaled_reward = reward / REWARD_SCALE
        sac_buffer.push(route_state, action_mask, action, scaled_reward, next_route_state, next_action_mask, done)
        
        # 7. Update Networks
        if len(sac_buffer) >= 1000:
            loss = sac_update(BATCH_SIZE)
        else:
            loss = None
            
        visited_arr = next_state_dict["visited"]
        current_node = next_state_dict["current_node"]
        
        if done:
            break
            
    avg_loss = sum(episode_loss) / len(episode_loss) if episode_loss else 0.0
    
    if total_travel_time < best_sac_time:
        best_sac_time = total_travel_time
        best_sac_route = env.history.copy()
        
    sac_history.append({"episode": episode, "travel_time": total_travel_time, "loss": avg_loss})
    
    print(f"SAC Episode {episode:02d}/{SAC_EPISODES} | Time: {total_travel_time:10.2f}s | Completed: {env.deliveries_completed:3d} | Loss: {avg_loss:.4f}")

print("\nSAC TRAINING COMPLETE")
print("Best SAC travel time:", best_sac_time, "seconds")

TRAINING TGCN + TRANSFORMER + SAC
SAC Episode 01/500 | Time:   40514.10s | Completed: 165 | Loss: 0.0000
SAC Episode 02/500 | Time:   41598.30s | Completed: 165 | Loss: 0.0000
SAC Episode 03/500 | Time:   40242.60s | Completed: 165 | Loss: 0.0000
SAC Episode 04/500 | Time:   40158.00s | Completed: 165 | Loss: 0.0000
SAC Episode 05/500 | Time:   41687.50s | Completed: 165 | Loss: 0.0000
SAC Episode 06/500 | Time:   38903.40s | Completed: 165 | Loss: 0.0000
SAC Episode 07/500 | Time:   41056.50s | Completed: 165 | Loss: 0.0000
SAC Episode 08/500 | Time:   37200.90s | Completed: 165 | Loss: 0.0000
SAC Episode 09/500 | Time:   39551.40s | Completed: 165 | Loss: 0.0000
SAC Episode 10/500 | Time:   40065.20s | Completed: 165 | Loss: 0.0000
SAC Episode 11/500 | Time:   39063.60s | Completed: 165 | Loss: 0.0000
SAC Episode 12/500 | Time:   38925.10s | Completed: 165 | Loss: 0.0000
SAC Episode 13/500 | Time:   38683.90s | Completed: 165 | Loss: 0.0000
SAC Episode 14/500 | Time:   40158.80s | Co

In [ ]:
# ==========================================================
# CELL 68 — Evaluation & Architecture Comparison
# ==========================================================
import pandas as pd

print("EVALUATION & COMPARISON")
print("=" * 60) 

sac_model.eval()
dynamic_encoder.eval()
temporal_encoder_v2.eval()

state_dict = env.reset()
current_node = state_dict["current_node"]
visited_arr = state_dict["visited"]
sac_eval_time = 0

# Deterministic Evaluation Run
with torch.no_grad():
    for step in range(len(node_ids) - 1):
        dynamic_node_state = dynamic_encoder(
            node_features=X.to(device), travel_features=travel_time_state.to(device),
            visited=visited_arr, current_node=current_node, adjacency=A_normalized.to(device)
        )
        route_state = temporal_encoder_v2(dynamic_node_state.unsqueeze(0))
        action_mask = torch.tensor(1.0 - visited_arr, dtype=torch.float32, device=device).unsqueeze(0)
        
        action = sac_model.select_action(route_state, action_mask, deterministic=True)
        next_state_dict, reward, done = env.step(action)
        
        sac_eval_time -= float(reward)
        visited_arr = next_state_dict["visited"]
        current_node = next_state_dict["current_node"]
        
        if done: break

# Fetching baseline stats from your earlier environment runs
ppo_eval_time = best_eval_time if 'best_eval_time' in globals() else 40769.5  # Fallback to known notebook value
nearest_neighbor_time = total_travel_time if 'total_travel_time' in globals() else 8386.8 # Fallback to known notebook value

data = {
    "Architecture": ["Nearest Neighbor (Baseline)", "TGCN + LSTM + PPO", "TGCN + Transformer + SAC (Fallback)"],
    "Total Travel Time (s)": [nearest_neighbor_time, ppo_eval_time, sac_eval_time],
    "Total Travel Time (min)": [nearest_neighbor_time / 60.0, ppo_eval_time / 60.0, sac_eval_time / 60.0]
}

comparison_df = pd.DataFrame(data)
comparison_df["% Difference vs PPO"] = ((comparison_df["Total Travel Time (s)"] - ppo_eval_time) / ppo_eval_time) * 100

print(comparison_df.to_string(index=False))

if sac_eval_time < ppo_eval_time:
    print(f"\n✓ Fallback architecture (Transformer + SAC) outperformed PPO by {abs(sac_eval_time - ppo_eval_time):.2f}s")
else:
    print(f"\n✓ Fallback architecture ran successfully, though PPO performed better by {abs(ppo_eval_time - sac_eval_time):.2f}s in this run.")

EVALUATION & COMPARISON
                       Architecture  Total Travel Time (s)  Total Travel Time (min)  % Difference vs PPO
        Nearest Neighbor (Baseline)           40457.199981               674.286666            -1.016813
                  TGCN + LSTM + PPO           40872.799859               681.213331             0.000000
TGCN + Transformer + SAC (Fallback)           40334.499947               672.241666            -1.317013

✓ Fallback architecture (Transformer + SAC) outperformed PPO by 538.30s
